## Here we are we cleaning the dataset. Some of the cleaning is already done in excel.

### I found that the dataset has negative values. Seeing the nature of it, it seems like refunds. So i am going to delete that rows. of negative and posiive values.

In [ ]:
import pandas as pd

# Load this file
df = pd.read_csv("eden_datasets/UL_EDEN_final_clean_transactions.csv")

# Convert date correctly for this file
df["TransDate"] = pd.to_datetime(
    df["TransDate"],
    format="%m/%d/%y %H:%M",
    errors="coerce"
)

print("Invalid TransDate rows:", df["TransDate"].isnull().sum())
print("Start date:", df["TransDate"].min())
print("End date:", df["TransDate"].max())

# Create date features
df["Date"] = df["TransDate"].dt.date
df["Hour"] = df["TransDate"].dt.hour
df["DayOfWeek"] = df["TransDate"].dt.day_name()
df["Month"] = df["TransDate"].dt.month
df["WeekOfYear"] = df["TransDate"].dt.isocalendar().week.astype(int)

df.head()

/var/folders/61/pw_dwqt140ndz64pvx21l9040000gn/T/ipykernel_860/3827497064.py:10: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df["TransDate"] = pd.to_datetime(df["TransDate"], dayfirst=True, errors="coerce")


Refund correction completed.
Original rows: 154978
Rows removed: 88
Final rows: 154890
Saved file as: UL_EDEN_refund_corrected_transactions.csv


### Now i am just auditing the done code

In [3]:
# Save removed refund-pair rows for checking/audit
refund_pair_audit = df[df.index.isin(rows_to_remove)].copy()

refund_pair_audit = refund_pair_audit.drop(
    columns=["RowID", "TransValueRounded", "AbsTransValue"]
)

audit_file = "UL_EDEN_removed_refund_pairs_audit.csv"
refund_pair_audit.to_csv(audit_file, index=False)

print("Audit file saved as:", audit_file)
print("Rows in audit file:", len(refund_pair_audit))

Audit file saved as: UL_EDEN_removed_refund_pairs_audit.csv
Rows in audit file: 88


In [4]:
df_check = pd.read_csv("UL_EDEN_refund_corrected_transactions.csv")

negative_rows_left = df_check[df_check["TransValue"] < 0]

print("Negative rows still left:", len(negative_rows_left))

negative_rows_left.head()

Negative rows still left: 810


,RECEIPT,TransDate,TransValue,PLUName,GroupCode,GroupName,PLUCode
72616,362234,2025-10-14 10:18:00,-0.15,OWN KEEP CUP -0.15,1,HOT BEVS,4262801
72617,362234,2025-10-14 10:18:00,-0.15,OWN KEEP CUP -0.15,1,HOT BEVS,4262801
73215,362543,2025-10-14 13:36:00,-0.15,OWN KEEP CUP -0.15,1,HOT BEVS,4262801
73249,362563,2025-10-14 13:52:00,-0.15,OWN KEEP CUP -0.15,1,HOT BEVS,4262801
73400,362653,2025-10-14 15:00:00,-0.15,OWN KEEP CUP -0.15,1,HOT BEVS,4262801


### foud that there is a transaction -ve that is called own keep cup. Have asled the data proider for clarification but until then I am assuming it is the customer using their own cup so dicount is given for that and that is not needed for forcasting order. Unless there is change in the scope to reduce economical expenses.

In [ ]:
import pandas as pd

df = pd.read_csv("eden_datasets/UL_EDEN_refund_corrected_transactions.csv")

# Separate OWN KEEP CUP rows for audit
own_cup_rows = df[df["PLUName"].str.contains("OWN KEEP CUP", case=False, na=False)].copy()

# Remove OWN KEEP CUP rows from final demand transaction dataset
df_final = df[~df["PLUName"].str.contains("OWN KEEP CUP", case=False, na=False)].copy()

# Save own cup rows separately
own_cup_rows.to_csv("UL_EDEN_own_cup_discount_rows_audit.csv", index=False)

# Save final cleaned transaction file
df_final.to_csv("UL_EDEN_final_clean_transactions.csv", index=False)

print("Original rows:", len(df))
print("OWN KEEP CUP rows removed:", len(own_cup_rows))
print("Final rows:", len(df_final))
print("Negative values remaining:", len(df_final[df_final["TransValue"] < 0]))

Original rows: 154890
OWN KEEP CUP rows removed: 810
Final rows: 154080
Negative values remaining: 0


### We need to check if there is any blnk missing values in the code

In [8]:
df = pd.read_csv("eden_datasets/UL_EDEN_final_clean_transactions.csv")

# Check missing values in all columns
print(df.isnull().sum())

RECEIPT       0
TransDate     0
TransValue    0
PLUName       0
GroupCode     0
GroupName     0
PLUCode       0
dtype: int64


In [9]:
important_columns = ["TransDate", "TransValue", "PLUName", "GroupName", "PLUCode"]

print(df[important_columns].isnull().sum())

TransDate     0
TransValue    0
PLUName       0
GroupName     0
PLUCode       0
dtype: int64


In [10]:
for col in important_columns:
    blank_count = df[col].astype(str).str.strip().eq("").sum()
    print(col, "blank values:", blank_count)

TransDate blank values: 0
TransValue blank values: 0
PLUName blank values: 0
GroupName blank values: 0
PLUCode blank values: 0


### we need to change the TransDate to more featues like time, day of the week, mnth etc

In [16]:
import pandas as pd

# Load this file
df = pd.read_csv("eden_datasets/UL_EDEN_final_clean_transactions.csv")

# Convert date correctly for this file
df["TransDate"] = pd.to_datetime(
    df["TransDate"],
    format="%m/%d/%y %H:%M",
    errors="coerce"
)

print("Invalid TransDate rows:", df["TransDate"].isnull().sum())
print("Start date:", df["TransDate"].min())
print("End date:", df["TransDate"].max())

# Create date features
df["Date"] = df["TransDate"].dt.date
df["Hour"] = df["TransDate"].dt.hour
df["DayOfWeek"] = df["TransDate"].dt.day_name()
df["Month"] = df["TransDate"].dt.month
df["WeekOfYear"] = df["TransDate"].dt.isocalendar().week.astype(int)

df.head()

Invalid TransDate rows: 0
Start date: 2025-04-01 08:02:00
End date: 2026-03-30 15:42:00


,RECEIPT,TransDate,TransValue,PLUName,GroupCode,GroupName,PLUCode,Date,Hour,DayOfWeek,Month,WeekOfYear
0,194814,2025-07-16 14:46:00,1026.0,VEGT MAINS 3,7,DINNER,4241483,2025-07-16,14,Wednesday,7,29
1,194832,2025-07-17 14:14:00,549.0,MAINS 3,7,DINNER,4241480,2025-07-17,14,Thursday,7,29
2,194782,2025-07-09 14:27:00,513.0,MAINS 3,7,DINNER,4241480,2025-07-09,14,Wednesday,7,28
3,194837,2025-07-18 14:35:00,513.0,MAINS 3,7,DINNER,4241480,2025-07-18,14,Friday,7,29
4,194801,2025-07-10 14:51:00,468.0,MAINS 3,7,DINNER,4241480,2025-07-10,14,Thursday,7,28


In [17]:
# Save corrected dataset with date features
df.to_csv("eden_datasets/UL_EDEN_final_clean_transactions_with_correct_dates.csv", index=False)

In [18]:
df["TransactionID"] = (
    df["RECEIPT"].astype(str).str.strip()
    + "_"
    + df["TransDate"].dt.strftime("%Y-%m-%d_%H-%M-%S")
)

# Save audit version with receipt
df.to_csv("eden_datasets/UL_EDEN_clean_with_transaction_id_audit.csv", index=False)

# Save model-ready version without receipt
df_model_ready = df.drop(columns=["RECEIPT"])

df_model_ready.to_csv("eden_datasets/UL_EDEN_clean_model_ready_transactions.csv", index=False)

print("Rows:", len(df_model_ready))
print("Unique TransactionIDs:", df_model_ready["TransactionID"].nunique())

Rows: 154080
Unique TransactionIDs: 96099


### in the dataset there is no information of how many units sold. So assuming until the data provider confirm, that eac row is one unit sold. we are adding that feature as well.

In [26]:
import pandas as pd

# Load the dataset with date features
df = pd.read_csv("eden_datasets/UL_EDEN_clean_model_ready_transactions.csv")

# Add UnitSold column
# Each row represents one sold unit/item
df["UnitSold"] = 1

# Save as a new dataset
df.to_csv("eden_datasets/UL_EDEN_transactions_with_date_features_and_unitsold.csv", index=False)

print("UnitSold column added successfully.")
print("Rows:", len(df))
df.head()

UnitSold column added successfully.
Rows: 154080


,TransDate,TransValue,PLUName,GroupCode,GroupName,PLUCode,Date,Hour,DayOfWeek,Month,WeekOfYear,TransactionID,UnitSold
0,2025-07-16 14:46:00,1026.0,VEGT MAINS 3,7,DINNER,4241483,2025-07-16,14,Wednesday,7,29,194814_2025-07-16_14-46-00,1
1,2025-07-17 14:14:00,549.0,MAINS 3,7,DINNER,4241480,2025-07-17,14,Thursday,7,29,194832_2025-07-17_14-14-00,1
2,2025-07-09 14:27:00,513.0,MAINS 3,7,DINNER,4241480,2025-07-09,14,Wednesday,7,28,194782_2025-07-09_14-27-00,1
3,2025-07-18 14:35:00,513.0,MAINS 3,7,DINNER,4241480,2025-07-18,14,Friday,7,29,194837_2025-07-18_14-35-00,1
4,2025-07-10 14:51:00,468.0,MAINS 3,7,DINNER,4241480,2025-07-10,14,Thursday,7,28,194801_2025-07-10_14-51-00,1


In [27]:
df[["TransDate", "PLUName", "TransValue", "UnitSold"]].head(10)

,TransDate,PLUName,TransValue,UnitSold
0,2025-07-16 14:46:00,VEGT MAINS 3,1026.0,1
1,2025-07-17 14:14:00,MAINS 3,549.0,1
2,2025-07-09 14:27:00,MAINS 3,513.0,1
3,2025-07-18 14:35:00,MAINS 3,513.0,1
4,2025-07-10 14:51:00,MAINS 3,468.0,1
5,2025-07-24 12:53:00,VEGT MAINS 3,468.0,1
6,2025-08-01 13:30:00,MAINS 3,450.0,1
7,2025-07-22 13:21:00,MAINS 3,441.0,1
8,2025-07-22 13:21:00,MAINS 3,441.0,1
9,2025-07-31 13:13:00,MAINS 3,432.0,1


### from the data provider I understoof that the own cup is indeed a discount for bringing their own cups. So we dont need that. It will be better to delete that discounts for a cleaner data set

### since that was all done with assumption we are going to the next step that is to delete the rows of DRS 15c which is the can deposit scheme. we dont need that.

In [28]:
import pandas as pd

# Load your current clean dataset
df = pd.read_csv("eden_datasets/UL_EDEN_transactions_with_date_features_and_unitsold.csv")

# Find DRS 15C rows
drs_rows = df[df["PLUName"].str.contains("DRS 15C", case=False, na=False)].copy()

# Remove DRS 15C rows
df_no_drs = df[~df["PLUName"].str.contains("DRS 15C", case=False, na=False)].copy()

# Save removed DRS rows for audit
drs_rows.to_csv("eden_datasets/UL_EDEN_DRS_15C_removed_audit.csv", index=False)

# Save new cleaned dataset
df_no_drs.to_csv("eden_datasets/UL_EDEN_clean_no_negative_no_owncup_no_drs.csv", index=False)

print("DRS 15C removal completed.")
print("Original rows:", len(df))
print("DRS 15C rows removed:", len(drs_rows))
print("Final rows:", len(df_no_drs))
print("Negative values remaining:", len(df_no_drs[df_no_drs["TransValue"] < 0]))

DRS 15C removal completed.
Original rows: 154080
DRS 15C rows removed: 15097
Final rows: 138983
Negative values remaining: 0


In [30]:
drs_rows[[ "TransDate", "TransValue", "PLUName", "GroupName", "PLUCode"]].head(20)

,TransDate,TransValue,PLUName,GroupName,PLUCode
138952,2025-04-01 10:43:00,0.15,DRS 15C,Confectionary,100000015
138953,2025-04-01 10:49:00,0.15,DRS 15C,Confectionary,100000015
138954,2025-04-01 11:52:00,0.15,DRS 15C,Confectionary,100000015
138955,2025-04-01 11:52:00,0.15,DRS 15C,Confectionary,100000015
138956,2025-04-01 12:02:00,0.15,DRS 15C,Confectionary,100000015
138957,2025-04-01 12:05:00,0.15,DRS 15C,Confectionary,100000015
138958,2025-04-01 12:06:00,0.15,DRS 15C,Confectionary,100000015
138959,2025-04-01 12:07:00,0.15,DRS 15C,Confectionary,100000015
138960,2025-04-01 12:12:00,0.15,DRS 15C,Confectionary,100000015
138961,2025-04-01 12:13:00,0.15,DRS 15C,Confectionary,100000015


In [31]:
df_check = pd.read_csv("eden_datasets/UL_EDEN_clean_no_negative_no_owncup_no_drs.csv")

print("DRS 15C rows left:", len(
    df_check[df_check["PLUName"].str.contains("DRS 15C", case=False, na=False)]
))

DRS 15C rows left: 0


### OPEN UL was confirmed by the data provoder as a miscellanous purchase by the unversity of limerick for special orders and thre for we dont need it in ingredient mappin but it is better to put it in demand forcasting prediction tho. But the ingredient mapping can be done by the restaurant based on the forecast prediction we give.

In [32]:
import pandas as pd
import glob
import os
from collections import Counter

# ============================================================
# 1. Find all CSV datasets made so far
# ============================================================

csv_files = sorted(set(glob.glob("*.csv") + glob.glob("eden_datasets/*.csv")))

print("CSV files found:")
for f in csv_files:
    print("-", f)


# ============================================================
# 2. Helper functions
# ============================================================

EXPECTED_START = pd.Timestamp("2025-04-01")
EXPECTED_END = pd.Timestamp("2026-03-30 23:59:59")

def parse_transdate(series):
    """
    Robust parser for the formats we have seen so far.
    It tries multiple formats without permanently changing the original data.
    """
    s = series.astype(str).str.strip()

    parsed = pd.Series(pd.NaT, index=s.index, dtype="datetime64[ns]")

    formats = [
        "%m/%d/%y %H:%M",       # 7/16/25 14:46
        "%d/%m/%y %H:%M",       # 16/7/25 14:46
        "%Y-%m-%d %H:%M:%S",    # 2025-07-16 14:46:00
        "%Y-%m-%d %H:%M",       # 2025-07-16 14:46
    ]

    for fmt in formats:
        mask = parsed.isna()
        parsed.loc[mask] = pd.to_datetime(s.loc[mask], format=fmt, errors="coerce")

    return parsed


def count_contains(df, col, text):
    if col not in df.columns:
        return None
    return df[col].astype(str).str.contains(text, case=False, na=False).sum()


def blank_count(df, col):
    if col not in df.columns:
        return None
    return df[col].astype(str).str.strip().eq("").sum()


def safe_sum(df, col):
    if col not in df.columns:
        return None
    return df[col].sum()


def audit_one_file(file_path):
    df = pd.read_csv(file_path)

    result = {
        "file": file_path,
        "rows": len(df),
        "cols": len(df.columns),
        "columns": list(df.columns),
    }

    # Missing / blank values
    key_cols = ["RECEIPT", "TransDate", "TransValue", "PLUName", "GroupCode", "GroupName", "PLUCode"]
    for col in key_cols:
        if col in df.columns:
            result[f"{col}_missing"] = int(df[col].isnull().sum())
            result[f"{col}_blank"] = int(blank_count(df, col))

    # Date checks
    if "TransDate" in df.columns:
        parsed_dates = parse_transdate(df["TransDate"])
        result["invalid_dates"] = int(parsed_dates.isna().sum())
        result["start_date"] = parsed_dates.min()
        result["end_date"] = parsed_dates.max()

        if parsed_dates.notna().any():
            result["date_range_ok"] = (
                parsed_dates.min() >= EXPECTED_START and
                parsed_dates.max() <= EXPECTED_END
            )
        else:
            result["date_range_ok"] = False
    else:
        result["invalid_dates"] = None
        result["start_date"] = None
        result["end_date"] = None
        result["date_range_ok"] = False

    # Negative / special rows
    if "TransValue" in df.columns:
        result["negative_rows"] = int((df["TransValue"] < 0).sum())
        result["total_value"] = round(float(df["TransValue"].sum()), 2)
    else:
        result["negative_rows"] = None
        result["total_value"] = None

    result["own_keep_cup_rows"] = count_contains(df, "PLUName", "OWN KEEP CUP")
    result["drs_15c_rows"] = count_contains(df, "PLUName", "DRS 15C")
    result["open_ul_rows"] = count_contains(df, "PLUName", "OPEN UL")
    result["kimbock_rows"] = count_contains(df, "PLUName", "KIMBOCK")
    result["kimbox_rows"] = count_contains(df, "PLUName", "KIMBOX")

    # Feature columns
    result["has_Date"] = "Date" in df.columns
    result["has_Hour"] = "Hour" in df.columns
    result["has_DayOfWeek"] = "DayOfWeek" in df.columns
    result["has_Month"] = "Month" in df.columns
    result["has_WeekOfYear"] = "WeekOfYear" in df.columns
    result["has_TransactionID"] = "TransactionID" in df.columns
    result["has_UnitSold"] = "UnitSold" in df.columns
    result["has_RECEIPT"] = "RECEIPT" in df.columns

    if "TransactionID" in df.columns:
        result["unique_transaction_ids"] = df["TransactionID"].nunique()
    else:
        result["unique_transaction_ids"] = None

    # Product text issues
    if "PLUName" in df.columns:
        plu = df["PLUName"].astype(str)

        result["leading_trailing_space_PLUName"] = int((plu != plu.str.strip()).sum())
        result["double_space_PLUName"] = int(plu.str.contains(r"\s{2,}", regex=True).sum())
        result["bad_encoding_rows"] = int(plu.str.contains("Â|â", regex=True, na=False).sum())
        result["unique_PLUNames"] = df["PLUName"].nunique()
    else:
        result["leading_trailing_space_PLUName"] = None
        result["double_space_PLUName"] = None
        result["bad_encoding_rows"] = None
        result["unique_PLUNames"] = None

    if "PLUCode" in df.columns:
        result["unique_PLUCodes"] = df["PLUCode"].nunique()
    else:
        result["unique_PLUCodes"] = None

    # Exact duplicate rows
    result["exact_duplicate_rows"] = int(df.duplicated().sum())

    return result


# ============================================================
# 3. Run audit for every CSV file
# ============================================================

audit_results = []

for file in csv_files:
    try:
        audit_results.append(audit_one_file(file))
    except Exception as e:
        audit_results.append({
            "file": file,
            "ERROR": str(e)
        })

audit_df = pd.DataFrame(audit_results)

display_cols = [
    "file",
    "rows",
    "cols",
    "invalid_dates",
    "start_date",
    "end_date",
    "date_range_ok",
    "negative_rows",
    "own_keep_cup_rows",
    "drs_15c_rows",
    "open_ul_rows",
    "has_TransactionID",
    "has_UnitSold",
    "has_RECEIPT",
    "unique_transaction_ids",
    "leading_trailing_space_PLUName",
    "double_space_PLUName",
    "bad_encoding_rows",
    "total_value"
]

display(audit_df[[c for c in display_cols if c in audit_df.columns]])


# ============================================================
# 4. Show files that are NOT safe because of wrong dates
# ============================================================

problem_dates = audit_df[audit_df["date_range_ok"] == False][
    ["file", "rows", "invalid_dates", "start_date", "end_date"]
]

print("\nFiles with date problems:")
display(problem_dates)


# ============================================================
# 5. Show best candidate files to continue from
# ============================================================

candidate_files = audit_df[
    (audit_df["date_range_ok"] == True) &
    (audit_df["negative_rows"] == 0) &
    (audit_df["own_keep_cup_rows"] == 0)
].copy()

print("\nBest candidate files to continue from:")
display(candidate_files[[
    "file",
    "rows",
    "start_date",
    "end_date",
    "drs_15c_rows",
    "has_TransactionID",
    "has_UnitSold",
    "has_RECEIPT"
]])

CSV files found:
- UL_EDEN_final_clean_transactions.csv
- UL_EDEN_own_cup_discount_rows_audit.csv
- UL_EDEN_transactions_with_date_features_and_unitsold.csv
- eden_datasets/UL EDEN Transaction 01.04.25 to 31.03.26.csv
- eden_datasets/UL_EDEN_DRS_15C_removed_audit.csv
- eden_datasets/UL_EDEN_clean_model_ready_transactions.csv
- eden_datasets/UL_EDEN_clean_no_negative_no_owncup_no_drs.csv
- eden_datasets/UL_EDEN_clean_with_transaction_id_audit.csv
- eden_datasets/UL_EDEN_demand_without_own_cup.csv
- eden_datasets/UL_EDEN_final_clean_transactions.csv
- eden_datasets/UL_EDEN_final_clean_transactions_with_correct_dates.csv
- eden_datasets/UL_EDEN_final_clean_transactions_with_dates.csv
- eden_datasets/UL_EDEN_own_cup_discount_rows_audit.csv
- eden_datasets/UL_EDEN_own_cup_removed_audit.csv
- eden_datasets/UL_EDEN_refund_corrected_transactions.csv
- eden_datasets/UL_EDEN_removed_refund_pairs_audit.csv
- eden_datasets/UL_EDEN_transactions_with_date_features_and_unitsold.csv


,file,rows,cols,invalid_dates,start_date,end_date,date_range_ok,negative_rows,own_keep_cup_rows,drs_15c_rows,open_ul_rows,has_TransactionID,has_UnitSold,has_RECEIPT,unique_transaction_ids,leading_trailing_space_PLUName,double_space_PLUName,bad_encoding_rows,total_value
0,UL_EDEN_final_clean_transactions.csv,154080,7,0,2025-04-01 08:02:00,2026-03-30 15:42:00,True,0,0,15097,24736,False,False,True,NaN,39,467,0,569659.89
1,UL_EDEN_own_cup_discount_rows_audit.csv,810,7,0,2025-10-14 10:18:00,2026-03-30 13:27:00,True,810,810,0,0,False,False,True,NaN,0,0,0,-121.50
2,UL_EDEN_transactions_with_date_features_and_un...,154080,13,0,2025-04-01 08:02:00,2026-03-30 15:42:00,True,0,0,15097,24736,True,True,False,96099.0,39,467,0,569659.89
3,eden_datasets/UL EDEN Transaction 01.04.25 to ...,154978,7,91718,2025-01-04 08:02:00,2026-12-03 16:07:00,False,854,812,15105,24756,False,False,True,NaN,39,469,0,569538.39
4,eden_datasets/UL_EDEN_DRS_15C_removed_audit.csv,15097,13,0,2025-04-01 10:43:00,2026-03-30 14:03:00,True,0,0,15097,0,True,True,False,14234.0,0,0,0,2264.55
5,eden_datasets/UL_EDEN_clean_model_ready_transa...,154080,12,0,2025-04-01 08:02:00,2026-03-30 15:42:00,True,0,0,15097,24736,True,False,False,96099.0,39,467,0,569659.89
6,eden_datasets/UL_EDEN_clean_no_negative_no_own...,138983,13,0,2025-04-01 08:02:00,2026-03-30 15:42:00,True,0,0,0,24736,True,True,False,96098.0,39,467,0,567395.34
7,eden_datasets/UL_EDEN_clean_with_transaction_i...,154080,13,0,2025-04-01 08:02:00,2026-03-30 15:42:00,True,0,0,15097,24736,True,False,True,96099.0,39,467,0,569659.89
8,eden_datasets/UL_EDEN_demand_without_own_cup.csv,154080,13,0,2025-01-04 08:02:00,2026-12-03 16:07:00,False,0,0,15097,24736,False,True,True,NaN,39,467,0,569659.89
9,eden_datasets/UL_EDEN_final_clean_transactions...,154080,7,0,2025-04-01 08:02:00,2026-03-30 15:42:00,True,0,0,15097,24736,False,False,True,NaN,39,467,0,569659.89



Files with date problems:


,file,rows,invalid_dates,start_date,end_date
3,eden_datasets/UL EDEN Transaction 01.04.25 to ...,154978,91718,2025-01-04 08:02:00,2026-12-03 16:07:00
8,eden_datasets/UL_EDEN_demand_without_own_cup.csv,154080,0,2025-01-04 08:02:00,2026-12-03 16:07:00
11,eden_datasets/UL_EDEN_final_clean_transactions...,154080,0,2025-01-04 08:02:00,2026-12-03 16:07:00
13,eden_datasets/UL_EDEN_own_cup_removed_audit.csv,0,0,NaT,NaT



Best candidate files to continue from:


,file,rows,start_date,end_date,drs_15c_rows,has_TransactionID,has_UnitSold,has_RECEIPT
0,UL_EDEN_final_clean_transactions.csv,154080,2025-04-01 08:02:00,2026-03-30 15:42:00,15097,False,False,True
2,UL_EDEN_transactions_with_date_features_and_un...,154080,2025-04-01 08:02:00,2026-03-30 15:42:00,15097,True,True,False
4,eden_datasets/UL_EDEN_DRS_15C_removed_audit.csv,15097,2025-04-01 10:43:00,2026-03-30 14:03:00,15097,True,True,False
5,eden_datasets/UL_EDEN_clean_model_ready_transa...,154080,2025-04-01 08:02:00,2026-03-30 15:42:00,15097,True,False,False
6,eden_datasets/UL_EDEN_clean_no_negative_no_own...,138983,2025-04-01 08:02:00,2026-03-30 15:42:00,0,True,True,False
7,eden_datasets/UL_EDEN_clean_with_transaction_i...,154080,2025-04-01 08:02:00,2026-03-30 15:42:00,15097,True,False,True
9,eden_datasets/UL_EDEN_final_clean_transactions...,154080,2025-04-01 08:02:00,2026-03-30 15:42:00,15097,False,False,True
10,eden_datasets/UL_EDEN_final_clean_transactions...,154080,2025-04-01 08:02:00,2026-03-30 15:42:00,15097,False,False,True
16,eden_datasets/UL_EDEN_transactions_with_date_f...,154080,2025-04-01 08:02:00,2026-03-30 15:42:00,15097,True,True,False


### there seems to be some problems with doubl spacing I saw in the data


In [33]:
import pandas as pd

# Load your current best file
df = pd.read_csv("eden_datasets/UL_EDEN_clean_no_negative_no_owncup_no_drs.csv")

# Make sure PLUName is text
plu = df["PLUName"].astype(str)

# Check leading/trailing spaces
leading_trailing_mask = plu != plu.str.strip()

# Check double or multiple spaces inside product names
double_space_mask = plu.str.contains(r"\s{2,}", regex=True, na=False)

print("Rows with leading/trailing spaces in PLUName:", leading_trailing_mask.sum())
print("Rows with double/multiple internal spaces in PLUName:", double_space_mask.sum())

Rows with leading/trailing spaces in PLUName: 39
Rows with double/multiple internal spaces in PLUName: 467


In [34]:
# Show the affected product names before cleaning
print("Leading/trailing space product names:")
display(
    df.loc[leading_trailing_mask, ["PLUName", "PLUCode", "GroupName"]]
    .drop_duplicates()
    .sort_values("PLUName")
)

print("Double/multiple space product names:")
display(
    df.loc[double_space_mask, ["PLUName", "PLUCode", "GroupName"]]
    .drop_duplicates()
    .sort_values("PLUName")
)

Leading/trailing space product names:


,PLUName,PLUCode,GroupName
104975,BROWN SCONE,4241436,CAKES/PASTRIES


Double/multiple space product names:


,PLUName,PLUCode,GroupName
1,MAINS 3,4241480,DINNER
130928,TAYTO CHEESE,4241401,SNACKS


In [35]:
# Clean PLUName and GroupName spacing
df["PLUName"] = df["PLUName"].astype(str).str.strip()
df["PLUName"] = df["PLUName"].str.replace(r"\s+", " ", regex=True)

df["GroupName"] = df["GroupName"].astype(str).str.strip()
df["GroupName"] = df["GroupName"].str.replace(r"\s+", " ", regex=True)

# Save as final cleaned transaction-level dataset
df.to_csv("eden_datasets/UL_EDEN_clean_final_model_ready_transactions.csv", index=False)

print("Saved final cleaned file.")

Saved final cleaned file.


In [36]:
plu = df["PLUName"].astype(str)

print("Rows with leading/trailing spaces after cleaning:", (plu != plu.str.strip()).sum())
print("Rows with double/multiple internal spaces after cleaning:", plu.str.contains(r"\s{2,}", regex=True, na=False).sum())

Rows with leading/trailing spaces after cleaning: 0
Rows with double/multiple internal spaces after cleaning: 0


### In the dataset I found that there is some orders that seem like bulk orders. Like 9 euro Kimbock for 369. This is less likely a unit sold. that single had many units sold and that needs to be corrected. We can ignore OPEN UL here as that is basically a different order from UL

In [37]:
import pandas as pd
import numpy as np

# ============================================================
# Load final cleaned transaction-level dataset
# ============================================================

df = pd.read_csv("eden_datasets/UL_EDEN_clean_final_model_ready_transactions.csv")

# Make sure date is datetime
df["TransDate"] = pd.to_datetime(df["TransDate"], errors="coerce")

# Safety check
required_cols = [
    "TransDate", "TransValue", "PLUCode", "PLUName",
    "GroupCode", "GroupName", "TransactionID", "UnitSold"
]

missing = [col for col in required_cols if col not in df.columns]
if missing:
    raise ValueError(f"Missing columns: {missing}")

# Convert money to cents to avoid floating point errors
df["TransValueCents"] = (df["TransValue"].astype(float) * 100).round().astype(int)

# Exclude OPEN UL from quantity adjustment
is_open_ul = df["PLUName"].astype(str).str.upper().str.strip().eq("OPEN UL")

print("Rows loaded:", len(df))
print("OPEN UL rows excluded from adjustment:", is_open_ul.sum())

Rows loaded: 138983
OPEN UL rows excluded from adjustment: 24736


In [38]:
# ============================================================
# Infer unit price per product
# ============================================================

product_cols = ["PLUCode", "PLUName", "GroupCode", "GroupName"]

# Use only non-OPEN UL positive rows
price_source = df[
    (~is_open_ul) &
    (df["TransValueCents"] > 0)
].copy()

# Count how often each transaction value appears for each product
price_counts = (
    price_source
    .groupby(product_cols + ["TransValueCents"])
    .size()
    .reset_index(name="PriceCount")
)

# Prefer values under €50 as normal unit-price candidates
# This prevents bulk values like €305 or €369 becoming the unit price
price_counts["IsNormalPriceCandidate"] = price_counts["TransValueCents"] < 5000

def choose_unit_price(group):
    normal_candidates = group[group["IsNormalPriceCandidate"]].copy()
    
    if len(normal_candidates) > 0:
        chosen_pool = normal_candidates
    else:
        chosen_pool = group.copy()
    
    # Choose most frequent price; if tie, choose smaller value
    chosen = chosen_pool.sort_values(
        ["PriceCount", "TransValueCents"],
        ascending=[False, True]
    ).iloc[0]
    
    return pd.Series({
        "UnitPriceCents": chosen["TransValueCents"],
        "UnitPrice": chosen["TransValueCents"] / 100,
        "UnitPriceFrequency": chosen["PriceCount"]
    })

unit_price_ref = (
    price_counts
    .groupby(product_cols)
    .apply(choose_unit_price)
    .reset_index()
)

print("Unit price reference table created.")
print("Products with inferred unit prices:", len(unit_price_ref))

display(unit_price_ref.head())

Unit price reference table created.
Products with inferred unit prices: 235


/var/folders/61/pw_dwqt140ndz64pvx21l9040000gn/T/ipykernel_48233/2511274951.py:48: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(choose_unit_price)


,PLUCode,PLUName,GroupCode,GroupName,UnitPriceCents,UnitPrice,UnitPriceFrequency
0,1031,TRAY BAKE,10,CAKES/PASTRIES,350.0,3.5,361.0
1,1313,MINCE PIE,10,CAKES/PASTRIES,150.0,1.5,46.0
2,1724,MUFFIN,10,CAKES/PASTRIES,350.0,3.5,700.0
3,2516,PORRIDGE/TOPPING,4,BREAKFAST,300.0,3.0,94.0
4,9102,SAUSAGE,4,BREAKFAST,100.0,1.0,277.0


In [39]:
# ============================================================
# Merge unit price back into main dataset
# ============================================================

df = df.merge(
    unit_price_ref,
    on=product_cols,
    how="left"
)

# Keep original UnitSold before changing anything
df["UnitSold_Original"] = df["UnitSold"]

# Default: keep UnitSold as it already is
df["UnitSold_Adjusted"] = df["UnitSold"]

# Calculate whether transaction value is an exact multiple of inferred unit price
df["IsExactPriceMultiple"] = False
df["InferredQuantityFromPrice"] = 1

valid_price = (
    (~is_open_ul) &
    df["UnitPriceCents"].notna() &
    (df["UnitPriceCents"] > 0) &
    (df["TransValueCents"] > df["UnitPriceCents"])
)

# Because UnitPriceCents may be float after merge, convert safely
df.loc[valid_price, "UnitPriceCents"] = df.loc[valid_price, "UnitPriceCents"].round().astype(int)

# Exact multiple check using cents
exact_multiple = valid_price & (
    df["TransValueCents"] % df["UnitPriceCents"].astype("Int64") == 0
)

df.loc[exact_multiple, "IsExactPriceMultiple"] = True
df.loc[exact_multiple, "InferredQuantityFromPrice"] = (
    df.loc[exact_multiple, "TransValueCents"] /
    df.loc[exact_multiple, "UnitPriceCents"]
).astype(int)

# Only adjust where inferred quantity is greater than 1
multi_unit_mask = exact_multiple & (df["InferredQuantityFromPrice"] > 1)

df.loc[multi_unit_mask, "UnitSold_Adjusted"] = df.loc[multi_unit_mask, "InferredQuantityFromPrice"]

# Add clear flag
df["MultiUnitAdjustedFlag"] = multi_unit_mask

print("Rows adjusted from UnitSold = 1 to inferred quantity:", df["MultiUnitAdjustedFlag"].sum())

Rows adjusted from UnitSold = 1 to inferred quantity: 61


In [40]:
# ============================================================
# Create audit of adjusted rows
# ============================================================

adjustment_audit = df[df["MultiUnitAdjustedFlag"]].copy()

audit_cols = [
    "TransDate",
    "TransactionID",
    "PLUCode",
    "PLUName",
    "GroupCode",
    "GroupName",
    "TransValue",
    "UnitPrice",
    "UnitSold_Original",
    "UnitSold_Adjusted",
    "InferredQuantityFromPrice"
]

adjustment_audit = adjustment_audit[audit_cols].sort_values(
    ["TransValue", "PLUName"],
    ascending=[False, True]
)

display(adjustment_audit)

adjustment_audit.to_csv(
    "eden_datasets/unit_sold_adjustment_audit_excluding_open_ul.csv",
    index=False
)

print("Audit saved to: eden_datasets/unit_sold_adjustment_audit_excluding_open_ul.csv")
print("Adjusted rows:", len(adjustment_audit))

,TransDate,TransactionID,PLUCode,PLUName,GroupCode,GroupName,TransValue,UnitPrice,UnitSold_Original,UnitSold_Adjusted,InferredQuantityFromPrice
0,2025-07-16 14:46:00,194814_2025-07-16_14-46-00,4241483,VEGT MAINS 3,7,DINNER,1026.0,9.0,1,114,114
1,2025-07-17 14:14:00,194832_2025-07-17_14-14-00,4241480,MAINS 3,7,DINNER,549.0,9.0,1,61,61
2,2025-07-09 14:27:00,194782_2025-07-09_14-27-00,4241480,MAINS 3,7,DINNER,513.0,9.0,1,57,57
3,2025-07-18 14:35:00,194837_2025-07-18_14-35-00,4241480,MAINS 3,7,DINNER,513.0,9.0,1,57,57
4,2025-07-10 14:51:00,194801_2025-07-10_14-51-00,4241480,MAINS 3,7,DINNER,468.0,9.0,1,52,52
...,...,...,...,...,...,...,...,...,...,...,...
90,2025-07-25 12:51:00,194864_2025-07-25_12-51-00,2000000019,FULL FAT CAN,2,COLD BEVS,18.0,1.8,1,10,10
92,2025-08-29 10:51:00,330639_2025-08-29_10-51-00,4241438,SCONE & BUTTER JAM,10,CAKES/PASTRIES,15.6,2.6,1,6,6
148,2025-07-25 12:51:00,194864_2025-07-25_12-51-00,4241474,SOUP OF THE DAY,8,PIZZA,10.5,3.5,1,3,3
1009,2025-05-29 18:48:00,184434_2025-05-29_18-48-00,2000000019,FULL FAT CAN,2,COLD BEVS,9.0,1.8,1,5,5


Audit saved to: eden_datasets/unit_sold_adjustment_audit_excluding_open_ul.csv
Adjusted rows: 61


### there was a mistak I found out tha the maximu unit transaction was 9.3 in the dataset and I had put the threshold for bulk as equal to or greater than 50 so now fixing it

In [43]:
from pathlib import Path

import numpy as np
import pandas as pd


# ============================================================
# File locations
# ============================================================

DATA_FOLDER = Path("eden_datasets")

INPUT_FILE = (
    DATA_FOLDER
    / "UL_EDEN_clean_final_model_ready_transactions.csv"
)

FINAL_OUTPUT_FILE = (
    DATA_FOLDER
    / "UL_EDEN_clean_final_model_ready_transactions_unitsold_corrected.csv"
)

FULL_AUDIT_OUTPUT_FILE = (
    DATA_FOLDER
    / "UL_EDEN_transactions_unitsold_corrected_with_audit_columns.csv"
)

ADJUSTED_ROWS_AUDIT_FILE = (
    DATA_FOLDER
    / "unit_sold_adjustment_audit_above_9_30_excluding_open_ul.csv"
)

UNIT_PRICE_REFERENCE_FILE = (
    DATA_FOLDER
    / "unit_price_reference_for_bulk_transactions.csv"
)

UNIT_PRICE_EVIDENCE_FILE = (
    DATA_FOLDER
    / "unit_price_candidate_evidence_for_bulk_transactions.csv"
)

ADJUSTMENT_SUMMARY_FILE = (
    DATA_FOLDER
    / "unit_sold_adjustment_summary_above_9_30.csv"
)


# Strict threshold:
# €9.30 is allowed to remain one unit.
# Only values ABOVE €9.30 are examined as multi-unit purchases.
SINGLE_ITEM_MAX_EURO = 9.30
SINGLE_ITEM_MAX_CENTS = 930


# Make sure the output folder exists
DATA_FOLDER.mkdir(parents=True, exist_ok=True)


if not INPUT_FILE.exists():
    raise FileNotFoundError(
        f"Input file was not found:\n{INPUT_FILE}\n\n"
        "Run the earlier notebook cell that creates "
        "'UL_EDEN_clean_final_model_ready_transactions.csv'."
    )


df = pd.read_csv(INPUT_FILE)

print("Dataset loaded successfully.")
print("Input file:", INPUT_FILE)
print("Rows:", len(df))
print("Columns:", len(df.columns))

display(df.head())

Dataset loaded successfully.
Input file: eden_datasets/UL_EDEN_clean_final_model_ready_transactions.csv
Rows: 138983
Columns: 13


,TransDate,TransValue,PLUName,GroupCode,GroupName,PLUCode,Date,Hour,DayOfWeek,Month,WeekOfYear,TransactionID,UnitSold
0,2025-07-16 14:46:00,1026.0,VEGT MAINS 3,7,DINNER,4241483,2025-07-16,14,Wednesday,7,29,194814_2025-07-16_14-46-00,1
1,2025-07-17 14:14:00,549.0,MAINS 3,7,DINNER,4241480,2025-07-17,14,Thursday,7,29,194832_2025-07-17_14-14-00,1
2,2025-07-09 14:27:00,513.0,MAINS 3,7,DINNER,4241480,2025-07-09,14,Wednesday,7,28,194782_2025-07-09_14-27-00,1
3,2025-07-18 14:35:00,513.0,MAINS 3,7,DINNER,4241480,2025-07-18,14,Friday,7,29,194837_2025-07-18_14-35-00,1
4,2025-07-10 14:51:00,468.0,MAINS 3,7,DINNER,4241480,2025-07-10,14,Thursday,7,28,194801_2025-07-10_14-51-00,1


In [44]:
# ============================================================
# Validate the exact input dataset
# ============================================================

required_columns = [
    "TransDate",
    "TransValue",
    "PLUName",
    "GroupCode",
    "GroupName",
    "PLUCode",
    "Date",
    "Hour",
    "DayOfWeek",
    "Month",
    "WeekOfYear",
    "TransactionID",
    "UnitSold",
]

missing_columns = [
    column
    for column in required_columns
    if column not in df.columns
]

if missing_columns:
    raise ValueError(
        f"Required columns are missing: {missing_columns}"
    )


# The exact uploaded dataset contains 138,983 rows.
if len(df) != 138_983:
    raise ValueError(
        f"Expected 138,983 rows, but the loaded file has "
        f"{len(df):,} rows. Check that the correct final "
        "cleaned dataset was loaded."
    )


# Save original structure and totals for later verification
original_columns = df.columns.tolist()
original_row_count = len(df)

original_transaction_value_cents = int(
    (
        pd.to_numeric(
            df["TransValue"],
            errors="raise"
        )
        * 100
    )
    .round()
    .sum()
)


# Convert data types safely
df["TransDate"] = pd.to_datetime(
    df["TransDate"],
    errors="raise"
)

df["TransValue"] = (
    pd.to_numeric(
        df["TransValue"],
        errors="raise"
    )
    .round(2)
)

df["UnitSold"] = (
    pd.to_numeric(
        df["UnitSold"],
        errors="raise"
    )
    .round()
    .astype("Int64")
)


# The uploaded input dataset should still have UnitSold = 1
if not df["UnitSold"].eq(1).all():
    raise ValueError(
        "The input file already contains UnitSold values "
        "other than 1. Load the original final cleaned "
        "transaction dataset before the bulk adjustment."
    )


# Clean text again as a safety measure
df["PLUName"] = (
    df["PLUName"]
    .astype("string")
    .str.strip()
    .str.replace(r"\s+", " ", regex=True)
)

df["GroupName"] = (
    df["GroupName"]
    .astype("string")
    .str.strip()
    .str.replace(r"\s+", " ", regex=True)
)


# Convert transaction values to cents
# This avoids floating-point calculation errors.
df["TransValueCents"] = (
    df["TransValue"] * 100
).round().astype("int64")


# Preserve original row order before merging later
df["_SourceRowOrder"] = np.arange(len(df))


print("Input validation passed.")
print("Rows:", f"{len(df):,}")
print("Original UnitSold total:", f"{df['UnitSold'].sum():,}")
print(
    "Transaction-value total:",
    f"€{df['TransValue'].sum():,.2f}"
)

Input validation passed.
Rows: 138,983
Original UnitSold total: 138,983
Transaction-value total: €567,395.34


In [45]:
# ============================================================
# Identify OPEN UL and possible multi-unit transactions
# ============================================================

is_open_ul = df["PLUName"].str.fullmatch(
    r"OPEN\s+UL",
    case=False,
    na=False
)


# Strictly greater than €9.30
bulk_transaction_mask = (
    ~is_open_ul
    & df["TransValueCents"].gt(SINGLE_ITEM_MAX_CENTS)
)


bulk_transaction_preview = (
    df.loc[
        bulk_transaction_mask,
        [
            "TransDate",
            "TransactionID",
            "PLUCode",
            "PLUName",
            "GroupCode",
            "GroupName",
            "TransValue",
            "UnitSold",
        ],
    ]
    .sort_values(
        ["TransValue", "PLUName"],
        ascending=[False, True]
    )
)


print("OPEN UL rows excluded:", f"{is_open_ul.sum():,}")

print(
    "Transactions above €9.30 excluding OPEN UL:",
    f"{bulk_transaction_mask.sum():,}"
)

print(
    "Affected products:",
    df.loc[
        bulk_transaction_mask,
        "PLUCode"
    ].nunique()
)


# Safety checks for the exact uploaded dataset
assert int(bulk_transaction_mask.sum()) == 59, (
    "Expected 59 transactions above €9.30 excluding "
    "OPEN UL. Check that the correct dataset was loaded."
)

assert (
    df.loc[
        bulk_transaction_mask,
        "PLUCode"
    ].nunique()
    == 11
), (
    "Expected 11 affected products."
)


display(bulk_transaction_preview)

OPEN UL rows excluded: 24,736
Transactions above €9.30 excluding OPEN UL: 59
Affected products: 11


,TransDate,TransactionID,PLUCode,PLUName,GroupCode,GroupName,TransValue,UnitSold
0,2025-07-16 14:46:00,194814_2025-07-16_14-46-00,4241483,VEGT MAINS 3,7,DINNER,1026.0,1
1,2025-07-17 14:14:00,194832_2025-07-17_14-14-00,4241480,MAINS 3,7,DINNER,549.0,1
2,2025-07-09 14:27:00,194782_2025-07-09_14-27-00,4241480,MAINS 3,7,DINNER,513.0,1
3,2025-07-18 14:35:00,194837_2025-07-18_14-35-00,4241480,MAINS 3,7,DINNER,513.0,1
4,2025-07-10 14:51:00,194801_2025-07-10_14-51-00,4241480,MAINS 3,7,DINNER,468.0,1
5,2025-07-24 12:53:00,194858_2025-07-24_12-53-00,4241483,VEGT MAINS 3,7,DINNER,468.0,1
6,2025-08-01 13:30:00,306448_2025-08-01_13-30-00,4241480,MAINS 3,7,DINNER,450.0,1
7,2025-07-22 13:21:00,194852_2025-07-22_13-21-00,4241480,MAINS 3,7,DINNER,441.0,1
8,2025-07-22 13:21:00,194852_2025-07-22_13-21-00,4241480,MAINS 3,7,DINNER,441.0,1
9,2025-07-31 13:13:00,194911_2025-07-31_13-13-00,4241480,MAINS 3,7,DINNER,432.0,1


In [46]:
# ============================================================
# Build product-specific unit-price reference
# ============================================================

product_columns = [
    "PLUCode",
    "PLUName",
    "GroupCode",
    "GroupName",
]


# Find the latest bulk-transaction date for each affected product
bulk_price_cutoffs = (
    df.loc[
        bulk_transaction_mask,
        product_columns + ["TransDate"],
    ]
    .groupby(
        product_columns,
        as_index=False,
        dropna=False
    )
    .agg(
        BulkPriceCutoffDate=("TransDate", "max")
    )
)


# Possible single-item prices:
# positive values up to and including €9.30
single_price_source = df.loc[
    (
        ~is_open_ul
        & df["TransValueCents"].between(
            1,
            SINGLE_ITEM_MAX_CENTS
        )
    ),
    product_columns
    + [
        "TransDate",
        "TransValueCents",
    ],
].copy()


# Retain only products that contain a bulk transaction
single_price_source = single_price_source.merge(
    bulk_price_cutoffs,
    on=product_columns,
    how="inner",
    validate="many_to_one"
)


# Do not use a later product price to explain an earlier bulk order
single_price_source = single_price_source.loc[
    single_price_source["TransDate"]
    <= single_price_source["BulkPriceCutoffDate"]
].copy()


# Count the frequency of each possible price
unit_price_candidate_evidence = (
    single_price_source
    .groupby(
        product_columns + ["TransValueCents"],
        as_index=False,
        dropna=False
    )
    .agg(
        UnitPriceFrequency=(
            "TransValueCents",
            "size"
        ),
        FirstSeen=(
            "TransDate",
            "min"
        ),
        LastSeen=(
            "TransDate",
            "max"
        ),
        BulkPriceCutoffDate=(
            "BulkPriceCutoffDate",
            "first"
        ),
    )
    .rename(
        columns={
            "TransValueCents": "UnitPriceCents"
        }
    )
)


unit_price_candidate_evidence["UnitPrice"] = (
    unit_price_candidate_evidence["UnitPriceCents"]
    / 100
)


# Select the most frequently observed price.
# In a frequency tie, select the smaller price.
unit_price_reference = (
    unit_price_candidate_evidence
    .sort_values(
        product_columns
        + [
            "UnitPriceFrequency",
            "UnitPriceCents",
        ],
        ascending=[
            True,
            True,
            True,
            True,
            False,
            True,
        ]
    )
    .drop_duplicates(
        subset=product_columns,
        keep="first"
    )
    .reset_index(drop=True)
)


unit_price_reference = unit_price_reference[
    product_columns
    + [
        "UnitPriceCents",
        "UnitPrice",
        "UnitPriceFrequency",
        "FirstSeen",
        "LastSeen",
        "BulkPriceCutoffDate",
    ]
]


if len(unit_price_reference) != 11:
    raise ValueError(
        "A unit price could not be identified for every "
        "affected product."
    )


print("Unit-price reference created.")
print(
    "Affected products with inferred prices:",
    len(unit_price_reference)
)

display(
    unit_price_reference.sort_values(
        "PLUName"
    )
)

Unit-price reference created.
Affected products with inferred prices: 11


,PLUCode,PLUName,GroupCode,GroupName,UnitPriceCents,UnitPrice,UnitPriceFrequency,FirstSeen,LastSeen,BulkPriceCutoffDate
4,4241430,BOX SALADS,6,SALADS,650,6.5,3,2025-06-11 12:38:00,2025-07-01 13:22:00,2025-07-28 13:26:00
2,4241425,CAFFE MOCHA BLISS BALLS,33,SNACKS,350,3.5,55,2025-04-28 13:32:00,2025-07-21 10:43:00,2025-07-21 11:54:00
1,3219121,CARTON OF WATER,2,COLD BEVS,220,2.2,601,2025-04-01 09:53:00,2025-08-01 10:06:00,2025-08-01 13:30:00
10,2000000019,FULL FAT CAN,2,COLD BEVS,180,1.8,789,2025-04-01 12:02:00,2025-07-25 12:51:00,2025-07-25 12:51:00
3,4241428,HAM & CHEESE SANDWICH CT,5,SANDWICHES,500,5.0,621,2025-04-01 10:36:00,2025-09-26 12:12:00,2025-09-26 13:20:00
7,4241476,KIMBOX MAINS 2,7,DINNER,700,7.0,3465,2025-04-01 12:04:00,2025-07-28 13:22:00,2025-07-28 13:26:00
8,4241480,MAINS 3,7,DINNER,900,9.0,28,2025-07-01 12:05:00,2025-07-31 13:13:00,2025-08-01 13:30:00
5,4241438,SCONE & BUTTER JAM,10,CAKES/PASTRIES,260,2.6,324,2025-04-01 10:00:00,2025-08-29 10:25:00,2025-08-29 10:51:00
6,4241474,SOUP OF THE DAY,8,PIZZA,350,3.5,219,2025-04-01 12:22:00,2025-07-31 13:39:00,2025-08-01 13:30:00
9,4241483,VEGT MAINS 3,7,DINNER,900,9.0,4,2025-07-01 13:03:00,2025-07-25 12:51:00,2025-07-25 12:51:00


In [47]:
# ============================================================
# Merge unit-price reference and calculate proposed quantities
# ============================================================

df = df.merge(
    unit_price_reference,
    on=product_columns,
    how="left",
    validate="many_to_one",
    sort=False
)


# Restore exact original row order after merge
df = (
    df.sort_values("_SourceRowOrder")
    .reset_index(drop=True)
)


# Convert merged integer fields to nullable integer type
df["UnitPriceCents"] = (
    pd.to_numeric(
        df["UnitPriceCents"],
        errors="coerce"
    )
    .round()
    .astype("Int64")
)

df["UnitPriceFrequency"] = (
    pd.to_numeric(
        df["UnitPriceFrequency"],
        errors="coerce"
    )
    .round()
    .astype("Int64")
)


# Recalculate masks after the merge
is_open_ul = df["PLUName"].str.fullmatch(
    r"OPEN\s+UL",
    case=False,
    na=False
)

bulk_transaction_mask = (
    ~is_open_ul
    & df["TransValueCents"].gt(SINGLE_ITEM_MAX_CENTS)
)


df["IsExactPriceMultiple"] = False

df["InferredQuantityFromPrice"] = pd.Series(
    pd.NA,
    index=df.index,
    dtype="Int64"
)


valid_bulk_price_mask = (
    bulk_transaction_mask
    & df["UnitPriceCents"].notna()
    & df["UnitPriceCents"].gt(0)
)


valid_bulk_indices = df.index[
    valid_bulk_price_mask
]


transaction_cents = df.loc[
    valid_bulk_indices,
    "TransValueCents"
].to_numpy(dtype="int64")

unit_price_cents = df.loc[
    valid_bulk_indices,
    "UnitPriceCents"
].to_numpy(dtype="int64")


exact_multiple_result = (
    transaction_cents % unit_price_cents
) == 0


exact_multiple_indices = valid_bulk_indices[
    exact_multiple_result
]


df.loc[
    exact_multiple_indices,
    "IsExactPriceMultiple"
] = True


df.loc[
    exact_multiple_indices,
    "InferredQuantityFromPrice"
] = (
    df.loc[
        exact_multiple_indices,
        "TransValueCents"
    ].to_numpy(dtype="int64")
    //
    df.loc[
        exact_multiple_indices,
        "UnitPriceCents"
    ].to_numpy(dtype="int64")
)


manual_review_mask = (
    bulk_transaction_mask
    & ~df["IsExactPriceMultiple"]
)


manual_review_rows = (
    df.loc[
        manual_review_mask,
        [
            "TransDate",
            "TransactionID",
            "PLUCode",
            "PLUName",
            "GroupName",
            "TransValue",
            "UnitPrice",
            "UnitPriceFrequency",
            "IsExactPriceMultiple",
        ],
    ]
    .sort_values(
        ["TransValue", "PLUName"],
        ascending=[False, True]
    )
)


print(
    "Transactions above €9.30:",
    int(bulk_transaction_mask.sum())
)

print(
    "Exact product-price multiples:",
    int(
        df.loc[
            bulk_transaction_mask,
            "IsExactPriceMultiple"
        ].sum()
    )
)

print(
    "Transactions requiring manual review:",
    len(manual_review_rows)
)


display(manual_review_rows)

Transactions above €9.30: 59
Exact product-price multiples: 59
Transactions requiring manual review: 0


,TransDate,TransactionID,PLUCode,PLUName,GroupName,TransValue,UnitPrice,UnitPriceFrequency,IsExactPriceMultiple


In [48]:
# ============================================================
# Do not continue if any bulk transaction is unresolved
# ============================================================

if len(manual_review_rows) > 0:
    raise ValueError(
        f"{len(manual_review_rows)} transaction(s) above "
        "€9.30 could not be matched exactly to the inferred "
        "product unit price. Review the displayed rows before "
        "changing UnitSold."
    )


assert int(bulk_transaction_mask.sum()) == 59

assert int(
    df.loc[
        bulk_transaction_mask,
        "IsExactPriceMultiple"
    ].sum()
) == 59


print(
    "All 59 transactions above €9.30 were resolved "
    "successfully."
)

All 59 transactions above €9.30 were resolved successfully.


In [49]:
# ============================================================
# Apply verified UnitSold adjustments
# ============================================================

# Keep the original value for the audit
df["UnitSold_Original"] = (
    df["UnitSold"]
    .copy()
    .astype("Int64")
)


# Default audit values
df["MultiUnitAdjustedFlag"] = False

df["UnitSoldAdjustmentMethod"] = (
    "Original UnitSold retained"
)


adjustment_mask = (
    bulk_transaction_mask
    & df["IsExactPriceMultiple"]
    & df["InferredQuantityFromPrice"].gt(1)
)


# Apply the inferred quantity directly to UnitSold
df.loc[
    adjustment_mask,
    "UnitSold"
] = df.loc[
    adjustment_mask,
    "InferredQuantityFromPrice"
]


df["UnitSold"] = (
    df["UnitSold"]
    .astype("Int64")
)


df.loc[
    adjustment_mask,
    "MultiUnitAdjustedFlag"
] = True


df.loc[
    adjustment_mask,
    "UnitSoldAdjustmentMethod"
] = (
    "Adjusted: TransValue > €9.30 and exact multiple "
    "of product-specific unit price"
)


# Explicitly document excluded OPEN UL rows
df.loc[
    is_open_ul,
    "UnitSoldAdjustmentMethod"
] = (
    "OPEN UL excluded; original UnitSold retained"
)


# Explicitly document normal transactions
df.loc[
    (
        ~is_open_ul
        & df["TransValueCents"].le(
            SINGLE_ITEM_MAX_CENTS
        )
    ),
    "UnitSoldAdjustmentMethod"
] = (
    "TransValue <= €9.30; original UnitSold retained"
)


print(
    "Rows adjusted:",
    int(df["MultiUnitAdjustedFlag"].sum())
)

print(
    "Rows not adjusted:",
    int(
        (~df["MultiUnitAdjustedFlag"]).sum()
    )
)

Rows adjusted: 59
Rows not adjusted: 138924


In [50]:
# ============================================================
# Create adjustment audit
# ============================================================

adjustment_audit_columns = [
    "TransDate",
    "TransactionID",
    "PLUCode",
    "PLUName",
    "GroupCode",
    "GroupName",
    "TransValue",
    "UnitPrice",
    "UnitPriceFrequency",
    "UnitSold_Original",
    "UnitSold",
    "InferredQuantityFromPrice",
    "IsExactPriceMultiple",
    "MultiUnitAdjustedFlag",
    "UnitSoldAdjustmentMethod",
]


adjustment_audit = (
    df.loc[
        adjustment_mask,
        adjustment_audit_columns,
    ]
    .sort_values(
        ["TransValue", "PLUName"],
        ascending=[False, True]
    )
    .reset_index(drop=True)
)


print("Adjusted rows:", len(adjustment_audit))

display(adjustment_audit)

Adjusted rows: 59


,TransDate,TransactionID,PLUCode,PLUName,GroupCode,GroupName,TransValue,UnitPrice,UnitPriceFrequency,UnitSold_Original,UnitSold,InferredQuantityFromPrice,IsExactPriceMultiple,MultiUnitAdjustedFlag,UnitSoldAdjustmentMethod
0,2025-07-16 14:46:00,194814_2025-07-16_14-46-00,4241483,VEGT MAINS 3,7,DINNER,1026.0,9.0,4,1,114,114,True,True,Adjusted: TransValue > €9.30 and exact multipl...
1,2025-07-17 14:14:00,194832_2025-07-17_14-14-00,4241480,MAINS 3,7,DINNER,549.0,9.0,28,1,61,61,True,True,Adjusted: TransValue > €9.30 and exact multipl...
2,2025-07-09 14:27:00,194782_2025-07-09_14-27-00,4241480,MAINS 3,7,DINNER,513.0,9.0,28,1,57,57,True,True,Adjusted: TransValue > €9.30 and exact multipl...
3,2025-07-18 14:35:00,194837_2025-07-18_14-35-00,4241480,MAINS 3,7,DINNER,513.0,9.0,28,1,57,57,True,True,Adjusted: TransValue > €9.30 and exact multipl...
4,2025-07-10 14:51:00,194801_2025-07-10_14-51-00,4241480,MAINS 3,7,DINNER,468.0,9.0,28,1,52,52,True,True,Adjusted: TransValue > €9.30 and exact multipl...
5,2025-07-24 12:53:00,194858_2025-07-24_12-53-00,4241483,VEGT MAINS 3,7,DINNER,468.0,9.0,4,1,52,52,True,True,Adjusted: TransValue > €9.30 and exact multipl...
6,2025-08-01 13:30:00,306448_2025-08-01_13-30-00,4241480,MAINS 3,7,DINNER,450.0,9.0,28,1,50,50,True,True,Adjusted: TransValue > €9.30 and exact multipl...
7,2025-07-22 13:21:00,194852_2025-07-22_13-21-00,4241480,MAINS 3,7,DINNER,441.0,9.0,28,1,49,49,True,True,Adjusted: TransValue > €9.30 and exact multipl...
8,2025-07-22 13:21:00,194852_2025-07-22_13-21-00,4241480,MAINS 3,7,DINNER,441.0,9.0,28,1,49,49,True,True,Adjusted: TransValue > €9.30 and exact multipl...
9,2025-07-31 13:13:00,194911_2025-07-31_13-13-00,4241480,MAINS 3,7,DINNER,432.0,9.0,28,1,48,48,True,True,Adjusted: TransValue > €9.30 and exact multipl...


In [51]:
# ============================================================
# Product-level adjustment summary
# ============================================================

adjustment_summary = (
    df.loc[
        adjustment_mask
    ]
    .groupby(
        [
            "PLUCode",
            "PLUName",
            "GroupCode",
            "GroupName",
            "UnitPrice",
        ],
        as_index=False,
        dropna=False
    )
    .agg(
        AdjustedTransactionRows=(
            "PLUCode",
            "size"
        ),
        MinimumTransactionValue=(
            "TransValue",
            "min"
        ),
        MaximumTransactionValue=(
            "TransValue",
            "max"
        ),
        MinimumInferredUnits=(
            "UnitSold",
            "min"
        ),
        MaximumInferredUnits=(
            "UnitSold",
            "max"
        ),
        OriginalUnitsAcrossRows=(
            "UnitSold_Original",
            "sum"
        ),
        CorrectedUnitsAcrossRows=(
            "UnitSold",
            "sum"
        ),
    )
)


adjustment_summary["AdditionalUnitsAdded"] = (
    adjustment_summary["CorrectedUnitsAcrossRows"]
    - adjustment_summary["OriginalUnitsAcrossRows"]
)


adjustment_summary = (
    adjustment_summary
    .sort_values(
        "AdditionalUnitsAdded",
        ascending=False
    )
    .reset_index(drop=True)
)


display(adjustment_summary)

,PLUCode,PLUName,GroupCode,GroupName,UnitPrice,AdjustedTransactionRows,MinimumTransactionValue,MaximumTransactionValue,MinimumInferredUnits,MaximumInferredUnits,OriginalUnitsAcrossRows,CorrectedUnitsAcrossRows,AdditionalUnitsAdded
0,4241474,SOUP OF THE DAY,8,PIZZA,3.5,15,10.5,399.0,3,114,15,719,704
1,4241428,HAM & CHEESE SANDWICH CT,5,SANDWICHES,5.0,10,305.0,305.0,61,61,10,610,600
2,4241480,MAINS 3,7,DINNER,9.0,9,207.0,549.0,23,61,9,446,437
3,4241483,VEGT MAINS 3,7,DINNER,9.0,4,405.0,1026.0,45,114,4,258,254
4,3219121,CARTON OF WATER,2,COLD BEVS,2.2,10,33.0,66.0,15,30,10,240,230
5,42534,€9 KIMBOCK,7,DINNER,9.0,2,369.0,369.0,41,41,2,82,80
6,2000000019,FULL FAT CAN,2,COLD BEVS,1.8,5,18.0,48.6,10,27,5,84,79
7,4241430,BOX SALADS,6,SALADS,6.5,1,318.5,318.5,49,49,1,49,48
8,4241476,KIMBOX MAINS 2,7,DINNER,7.0,1,343.0,343.0,49,49,1,49,48
9,4241425,CAFFE MOCHA BLISS BALLS,33,SNACKS,3.5,1,21.0,21.0,6,6,1,6,5


In [52]:
# ============================================================
# Final validation
# ============================================================

adjusted_row_count = int(
    adjustment_mask.sum()
)

original_unit_total = int(
    df["UnitSold_Original"].sum()
)

corrected_unit_total = int(
    df["UnitSold"].sum()
)

additional_units = (
    corrected_unit_total
    - original_unit_total
)


# 1. Row count must not change
assert len(df) == original_row_count, (
    "Row count changed during adjustment."
)


# 2. Exactly 59 rows should be adjusted
assert adjusted_row_count == 59, (
    f"Expected 59 adjusted rows, found "
    f"{adjusted_row_count}."
)


# 3. OPEN UL must never be adjusted
assert not df.loc[
    is_open_ul,
    "MultiUnitAdjustedFlag"
].any(), (
    "An OPEN UL row was adjusted."
)


# 4. No value of €9.30 or below can be adjusted
assert not (
    df["MultiUnitAdjustedFlag"]
    & df["TransValueCents"].le(
        SINGLE_ITEM_MAX_CENTS
    )
).any(), (
    "A transaction of €9.30 or below was adjusted."
)


# 5. All non-adjusted rows must retain original UnitSold
assert df.loc[
    ~adjustment_mask,
    "UnitSold"
].equals(
    df.loc[
        ~adjustment_mask,
        "UnitSold_Original"
    ]
), (
    "A non-bulk transaction's UnitSold was changed."
)


# 6. Reconstruct transaction value using quantity and price
reconstructed_transaction_cents = (
    df.loc[
        adjustment_mask,
        "UnitSold"
    ].to_numpy(dtype="int64")
    *
    df.loc[
        adjustment_mask,
        "UnitPriceCents"
    ].to_numpy(dtype="int64")
)


actual_transaction_cents = df.loc[
    adjustment_mask,
    "TransValueCents"
].to_numpy(dtype="int64")


assert np.array_equal(
    reconstructed_transaction_cents,
    actual_transaction_cents
), (
    "At least one UnitSold × UnitPrice calculation "
    "does not equal its TransValue."
)


# 7. Transaction value must not change
current_transaction_value_cents = int(
    df["TransValueCents"].sum()
)

assert (
    current_transaction_value_cents
    == original_transaction_value_cents
), (
    "The total transaction value changed."
)


# 8. No UnitSold can be lower than 1
assert df["UnitSold"].ge(1).all(), (
    "A UnitSold value is below 1."
)


# Exact expected results from the uploaded dataset
assert original_unit_total == 138_983

assert corrected_unit_total == 141_473

assert additional_units == 2_490


print("All validation checks passed.")
print()
print("Rows:", f"{len(df):,}")
print("Adjusted transaction rows:", f"{adjusted_row_count:,}")
print("Original UnitSold total:", f"{original_unit_total:,}")
print("Corrected UnitSold total:", f"{corrected_unit_total:,}")
print("Additional units identified:", f"{additional_units:,}")
print("OPEN UL adjusted:", 0)
print("Transactions <= €9.30 adjusted:", 0)

All validation checks passed.

Rows: 138,983
Adjusted transaction rows: 59
Original UnitSold total: 138,983
Corrected UnitSold total: 141,473
Additional units identified: 2,490
OPEN UL adjusted: 0
Transactions <= €9.30 adjusted: 0


In [53]:
# ============================================================
# Locate remaining FULL FAT CAN bulk transactions
# ============================================================

FULL_FAT_CAN_PLU_CODE = 2000000019
FULL_FAT_CAN_UNIT_PRICE_CENTS = 180
FULL_FAT_CAN_UNIT_PRICE = 1.80


# Recreate TransValueCents only if necessary
if "TransValueCents" not in df.columns:
    df["TransValueCents"] = (
        pd.to_numeric(
            df["TransValue"],
            errors="raise"
        )
        .mul(100)
        .round()
        .astype("int64")
    )


full_fat_can_mask = (
    pd.to_numeric(
        df["PLUCode"],
        errors="coerce"
    ).eq(FULL_FAT_CAN_PLU_CODE)
    & df["PLUName"]
        .astype("string")
        .str.strip()
        .str.fullmatch(
            r"FULL\s+FAT\s+CAN",
            case=False,
            na=False
        )
)


already_adjusted_flag = (
    df["MultiUnitAdjustedFlag"]
    .fillna(False)
    .astype(bool)
)


# Find FULL FAT CAN transactions that:
# 1. Are more than one €1.80 unit
# 2. Are €9.30 or below
# 3. Are exact multiples of €1.80
# 4. Have not already been adjusted
remaining_full_fat_can_mask = (
    full_fat_can_mask
    & df["TransValueCents"].gt(
        FULL_FAT_CAN_UNIT_PRICE_CENTS
    )
    & df["TransValueCents"].le(930)
    & (
        df["TransValueCents"]
        % FULL_FAT_CAN_UNIT_PRICE_CENTS
    ).eq(0)
    & ~already_adjusted_flag
)


remaining_full_fat_can_preview = df.loc[
    remaining_full_fat_can_mask,
    [
        "TransDate",
        "TransactionID",
        "PLUCode",
        "PLUName",
        "GroupName",
        "TransValue",
        "UnitSold_Original",
        "UnitSold",
        "MultiUnitAdjustedFlag",
    ],
].copy()


remaining_full_fat_can_preview[
    "ConfirmedUnitPrice"
] = FULL_FAT_CAN_UNIT_PRICE


remaining_full_fat_can_preview[
    "ProposedUnitSold"
] = (
    df.loc[
        remaining_full_fat_can_mask,
        "TransValueCents"
    ]
    // FULL_FAT_CAN_UNIT_PRICE_CENTS
).astype("int64")


remaining_full_fat_can_preview = (
    remaining_full_fat_can_preview
    .sort_values(
        "TransValue",
        ascending=False
    )
    .reset_index(drop=True)
)


print(
    "Remaining FULL FAT CAN rows:",
    len(remaining_full_fat_can_preview)
)

display(remaining_full_fat_can_preview)

Remaining FULL FAT CAN rows: 2


,TransDate,TransactionID,PLUCode,PLUName,GroupName,TransValue,UnitSold_Original,UnitSold,MultiUnitAdjustedFlag,ConfirmedUnitPrice,ProposedUnitSold
0,2025-05-29 18:48:00,184434_2025-05-29_18-48-00,2000000019,FULL FAT CAN,COLD BEVS,9.0,1,1,False,1.8,5
1,2025-07-24 12:53:00,194858_2025-07-24_12-53-00,2000000019,FULL FAT CAN,COLD BEVS,7.2,1,1,False,1.8,4


In [54]:
# ============================================================
# Verify the exact two transactions before changing anything
# ============================================================

expected_full_fat_can_corrections = {
    (
        "184434_2025-05-29_18-48-00",
        900,
    ): 5,

    (
        "194858_2025-07-24_12-53-00",
        720,
    ): 4,
}


observed_transaction_keys = set(
    zip(
        df.loc[
            remaining_full_fat_can_mask,
            "TransactionID"
        ].astype(str),

        df.loc[
            remaining_full_fat_can_mask,
            "TransValueCents"
        ].astype(int),
    )
)


expected_transaction_keys = set(
    expected_full_fat_can_corrections.keys()
)


assert observed_transaction_keys == expected_transaction_keys, (
    "The remaining FULL FAT CAN rows do not match the "
    "two expected transactions. No changes were made."
)


assert len(
    df.loc[
        remaining_full_fat_can_mask
    ]
) == 2, (
    "Expected exactly two remaining FULL FAT CAN rows."
)


assert df.loc[
    remaining_full_fat_can_mask,
    "UnitSold"
].eq(1).all(), (
    "One of the two rows has already been adjusted."
)


assert df.loc[
    remaining_full_fat_can_mask,
    "UnitSold_Original"
].eq(1).all(), (
    "Unexpected UnitSold_Original value detected."
)


calculated_quantities = (
    df.loc[
        remaining_full_fat_can_mask,
        "TransValueCents"
    ]
    // FULL_FAT_CAN_UNIT_PRICE_CENTS
).astype(int)


assert set(calculated_quantities) == {4, 5}, (
    "The calculated quantities are not the expected "
    "values of 4 and 5."
)


print("The two remaining transactions were verified.")
print("€9.00 / €1.80 = 5 units")
print("€7.20 / €1.80 = 4 units")

The two remaining transactions were verified.
€9.00 / €1.80 = 5 units
€7.20 / €1.80 = 4 units


In [55]:
# ============================================================
# Apply remaining FULL FAT CAN corrections
# ============================================================

# Record the confirmed unit price
df.loc[
    remaining_full_fat_can_mask,
    "UnitPriceCents"
] = FULL_FAT_CAN_UNIT_PRICE_CENTS


df.loc[
    remaining_full_fat_can_mask,
    "UnitPrice"
] = FULL_FAT_CAN_UNIT_PRICE


# Confirm that both are exact multiples
df.loc[
    remaining_full_fat_can_mask,
    "IsExactPriceMultiple"
] = True


# Calculate inferred units
df.loc[
    remaining_full_fat_can_mask,
    "InferredQuantityFromPrice"
] = (
    df.loc[
        remaining_full_fat_can_mask,
        "TransValueCents"
    ]
    // FULL_FAT_CAN_UNIT_PRICE_CENTS
).astype("int64")


# Apply inferred units to UnitSold
df.loc[
    remaining_full_fat_can_mask,
    "UnitSold"
] = df.loc[
    remaining_full_fat_can_mask,
    "InferredQuantityFromPrice"
]


# Mark both rows as adjusted
df.loc[
    remaining_full_fat_can_mask,
    "MultiUnitAdjustedFlag"
] = True


df.loc[
    remaining_full_fat_can_mask,
    "UnitSoldAdjustmentMethod"
] = (
    "Adjusted: historical FULL FAT CAN transaction "
    "is an exact multiple of the confirmed €1.80 unit price"
)


# Restore integer column types
df["UnitSold"] = (
    pd.to_numeric(
        df["UnitSold"],
        errors="raise"
    )
    .round()
    .astype("Int64")
)


df["InferredQuantityFromPrice"] = (
    pd.to_numeric(
        df["InferredQuantityFromPrice"],
        errors="coerce"
    )
    .round()
    .astype("Int64")
)


df["UnitPriceCents"] = (
    pd.to_numeric(
        df["UnitPriceCents"],
        errors="coerce"
    )
    .round()
    .astype("Int64")
)


print(
    "Additional FULL FAT CAN rows corrected:",
    int(remaining_full_fat_can_mask.sum())
)

Additional FULL FAT CAN rows corrected: 2


In [56]:
# ============================================================
# Inspect the two newly corrected rows
# ============================================================

new_full_fat_can_corrections = (
    df.loc[
        remaining_full_fat_can_mask,
        [
            "TransDate",
            "TransactionID",
            "PLUCode",
            "PLUName",
            "GroupName",
            "TransValue",
            "UnitPrice",
            "UnitSold_Original",
            "UnitSold",
            "InferredQuantityFromPrice",
            "IsExactPriceMultiple",
            "MultiUnitAdjustedFlag",
            "UnitSoldAdjustmentMethod",
        ],
    ]
    .sort_values(
        "TransValue",
        ascending=False
    )
    .reset_index(drop=True)
)


display(new_full_fat_can_corrections)

,TransDate,TransactionID,PLUCode,PLUName,GroupName,TransValue,UnitPrice,UnitSold_Original,UnitSold,InferredQuantityFromPrice,IsExactPriceMultiple,MultiUnitAdjustedFlag,UnitSoldAdjustmentMethod
0,2025-05-29 18:48:00,184434_2025-05-29_18-48-00,2000000019,FULL FAT CAN,COLD BEVS,9.0,1.8,1,5,5,True,True,Adjusted: historical FULL FAT CAN transaction ...
1,2025-07-24 12:53:00,194858_2025-07-24_12-53-00,2000000019,FULL FAT CAN,COLD BEVS,7.2,1.8,1,4,4,True,True,Adjusted: historical FULL FAT CAN transaction ...


In [57]:
# ============================================================
# Rebuild final adjustment mask
# ============================================================

adjustment_mask = (
    df["MultiUnitAdjustedFlag"]
    .fillna(False)
    .astype(bool)
)


print(
    "Total adjusted transaction rows:",
    int(adjustment_mask.sum())
)


assert int(adjustment_mask.sum()) == 61, (
    "Expected 61 total adjusted transaction rows: "
    "59 original corrections plus 2 FULL FAT CAN corrections."
)


print("Final adjustment mask updated successfully.")

Total adjusted transaction rows: 61
Final adjustment mask updated successfully.


In [58]:
# ============================================================
# Rebuild audit with all 61 adjusted rows
# ============================================================

adjustment_audit_columns = [
    "TransDate",
    "TransactionID",
    "PLUCode",
    "PLUName",
    "GroupCode",
    "GroupName",
    "TransValue",
    "UnitPrice",
    "UnitPriceFrequency",
    "UnitSold_Original",
    "UnitSold",
    "InferredQuantityFromPrice",
    "IsExactPriceMultiple",
    "MultiUnitAdjustedFlag",
    "UnitSoldAdjustmentMethod",
]


adjustment_audit = (
    df.loc[
        adjustment_mask,
        adjustment_audit_columns,
    ]
    .sort_values(
        [
            "TransValue",
            "PLUName",
        ],
        ascending=[
            False,
            True,
        ]
    )
    .reset_index(drop=True)
)


assert len(adjustment_audit) == 61, (
    "The rebuilt audit should contain 61 rows."
)


print(
    "Rebuilt adjustment-audit rows:",
    len(adjustment_audit)
)


display(
    adjustment_audit.loc[
        adjustment_audit["PLUName"].eq(
            "FULL FAT CAN"
        )
    ]
)

Rebuilt adjustment-audit rows: 61


,TransDate,TransactionID,PLUCode,PLUName,GroupCode,GroupName,TransValue,UnitPrice,UnitPriceFrequency,UnitSold_Original,UnitSold,InferredQuantityFromPrice,IsExactPriceMultiple,MultiUnitAdjustedFlag,UnitSoldAdjustmentMethod
46,2025-07-09 14:27:00,194782_2025-07-09_14-27-00,2000000019,FULL FAT CAN,2,COLD BEVS,48.6,1.8,789,1,27,27,True,True,Adjusted: TransValue > €9.30 and exact multipl...
47,2025-07-08 15:13:00,194769_2025-07-08_15-13-00,2000000019,FULL FAT CAN,2,COLD BEVS,46.8,1.8,789,1,26,26,True,True,Adjusted: TransValue > €9.30 and exact multipl...
54,2025-07-25 12:51:00,194864_2025-07-25_12-51-00,2000000019,FULL FAT CAN,2,COLD BEVS,19.8,1.8,789,1,11,11,True,True,Adjusted: TransValue > €9.30 and exact multipl...
55,2025-07-24 12:53:00,194858_2025-07-24_12-53-00,2000000019,FULL FAT CAN,2,COLD BEVS,18.0,1.8,789,1,10,10,True,True,Adjusted: TransValue > €9.30 and exact multipl...
56,2025-07-25 12:51:00,194864_2025-07-25_12-51-00,2000000019,FULL FAT CAN,2,COLD BEVS,18.0,1.8,789,1,10,10,True,True,Adjusted: TransValue > €9.30 and exact multipl...
59,2025-05-29 18:48:00,184434_2025-05-29_18-48-00,2000000019,FULL FAT CAN,2,COLD BEVS,9.0,1.8,789,1,5,5,True,True,Adjusted: historical FULL FAT CAN transaction ...
60,2025-07-24 12:53:00,194858_2025-07-24_12-53-00,2000000019,FULL FAT CAN,2,COLD BEVS,7.2,1.8,789,1,4,4,True,True,Adjusted: historical FULL FAT CAN transaction ...


In [59]:
# ============================================================
# Rebuild product-level adjustment summary
# ============================================================

adjustment_summary = (
    df.loc[
        adjustment_mask
    ]
    .groupby(
        [
            "PLUCode",
            "PLUName",
            "GroupCode",
            "GroupName",
            "UnitPrice",
        ],
        as_index=False,
        dropna=False
    )
    .agg(
        AdjustedTransactionRows=(
            "PLUCode",
            "size"
        ),
        MinimumTransactionValue=(
            "TransValue",
            "min"
        ),
        MaximumTransactionValue=(
            "TransValue",
            "max"
        ),
        MinimumInferredUnits=(
            "UnitSold",
            "min"
        ),
        MaximumInferredUnits=(
            "UnitSold",
            "max"
        ),
        OriginalUnitsAcrossRows=(
            "UnitSold_Original",
            "sum"
        ),
        CorrectedUnitsAcrossRows=(
            "UnitSold",
            "sum"
        ),
    )
)


adjustment_summary[
    "AdditionalUnitsAdded"
] = (
    adjustment_summary[
        "CorrectedUnitsAcrossRows"
    ]
    - adjustment_summary[
        "OriginalUnitsAcrossRows"
    ]
)


adjustment_summary = (
    adjustment_summary
    .sort_values(
        "AdditionalUnitsAdded",
        ascending=False
    )
    .reset_index(drop=True)
)


full_fat_can_summary = adjustment_summary.loc[
    adjustment_summary["PLUName"].eq(
        "FULL FAT CAN"
    )
]


display(full_fat_can_summary)

,PLUCode,PLUName,GroupCode,GroupName,UnitPrice,AdjustedTransactionRows,MinimumTransactionValue,MaximumTransactionValue,MinimumInferredUnits,MaximumInferredUnits,OriginalUnitsAcrossRows,CorrectedUnitsAcrossRows,AdditionalUnitsAdded
5,2000000019,FULL FAT CAN,2,COLD BEVS,1.8,7,7.2,48.6,4,27,7,93,86


In [63]:
# ============================================================
# New Cell 10H
# Final validation after all FULL FAT CAN corrections
# ============================================================

# Recreate TransValueCents if it is not currently available
if "TransValueCents" not in df.columns:
    df["TransValueCents"] = (
        pd.to_numeric(
            df["TransValue"],
            errors="raise"
        )
        .mul(100)
        .round()
        .astype("int64")
    )


# Rebuild the final adjustment mask from the audit flag
adjustment_mask = (
    df["MultiUnitAdjustedFlag"]
    .fillna(False)
    .astype(bool)
)


# Define the two confirmed historical FULL FAT CAN corrections.
#
# TransactionID alone is not unique because one receipt can
# contain several different product lines. Therefore, each row
# is identified using:
#
# TransactionID + PLUCode + TransValueCents

expected_low_value_corrections = [
    {
        "TransactionID": "184434_2025-05-29_18-48-00",
        "PLUCode": 2000000019,
        "TransValueCents": 900,
        "UnitSold": 5,
    },
    {
        "TransactionID": "194858_2025-07-24_12-53-00",
        "PLUCode": 2000000019,
        "TransValueCents": 720,
        "UnitSold": 4,
    },
]


# ------------------------------------------------------------
# 1. Validate the total number of transaction rows
# ------------------------------------------------------------

assert len(df) == original_row_count, (
    "ERROR: The number of transaction rows changed."
)


# ------------------------------------------------------------
# 2. Exactly 61 transaction rows must now be adjusted
# ------------------------------------------------------------

adjusted_row_count = int(
    adjustment_mask.sum()
)


assert adjusted_row_count == 61, (
    f"ERROR: Expected 61 adjusted transaction rows, "
    f"but found {adjusted_row_count}."
)


# ------------------------------------------------------------
# 3. OPEN UL must remain completely unchanged
# ------------------------------------------------------------

is_open_ul = (
    df["PLUName"]
    .astype("string")
    .str.strip()
    .str.fullmatch(
        r"OPEN\s+UL",
        case=False,
        na=False
    )
)


open_ul_adjusted_count = int(
    adjustment_mask.loc[
        is_open_ul
    ].sum()
)


assert open_ul_adjusted_count == 0, (
    "ERROR: At least one OPEN UL row was adjusted."
)


assert (
    df.loc[
        is_open_ul,
        "UnitSold"
    ]
    .astype("Int64")
    .equals(
        df.loc[
            is_open_ul,
            "UnitSold_Original"
        ].astype("Int64")
    )
), (
    "ERROR: At least one OPEN UL UnitSold value changed."
)


# ------------------------------------------------------------
# 4. Locate adjusted transactions at or below €9.30
# ------------------------------------------------------------

low_value_adjusted_mask = (
    adjustment_mask
    & df["TransValueCents"].le(930)
)


low_value_adjusted_rows = (
    df.loc[
        low_value_adjusted_mask,
        [
            "TransDate",
            "TransactionID",
            "PLUCode",
            "PLUName",
            "GroupName",
            "TransValue",
            "TransValueCents",
            "UnitPrice",
            "UnitPriceCents",
            "UnitSold_Original",
            "UnitSold",
            "InferredQuantityFromPrice",
            "IsExactPriceMultiple",
            "MultiUnitAdjustedFlag",
            "UnitSoldAdjustmentMethod",
        ],
    ]
    .sort_values(
        [
            "TransValue",
            "TransactionID",
        ],
        ascending=[
            False,
            True,
        ]
    )
    .reset_index(drop=True)
)


assert len(low_value_adjusted_rows) == 2, (
    "ERROR: Exactly two adjusted transaction rows at or "
    "below €9.30 were expected."
)


assert (
    low_value_adjusted_rows["PLUName"]
    .astype("string")
    .str.strip()
    .eq("FULL FAT CAN")
    .all()
), (
    "ERROR: A product other than FULL FAT CAN was adjusted "
    "at or below €9.30."
)


# ------------------------------------------------------------
# 5. Verify that the two observed rows are the expected rows
# ------------------------------------------------------------

observed_low_value_keys = set(
    zip(
        low_value_adjusted_rows[
            "TransactionID"
        ].astype(str),

        pd.to_numeric(
            low_value_adjusted_rows["PLUCode"],
            errors="raise"
        ).astype("int64"),

        pd.to_numeric(
            low_value_adjusted_rows["TransValueCents"],
            errors="raise"
        ).astype("int64"),
    )
)


expected_low_value_keys = {
    (
        expected["TransactionID"],
        expected["PLUCode"],
        expected["TransValueCents"],
    )
    for expected in expected_low_value_corrections
}


assert observed_low_value_keys == expected_low_value_keys, (
    "ERROR: The adjusted low-value product lines do not "
    "match the two expected FULL FAT CAN corrections."
)


# ------------------------------------------------------------
# 6. Validate each historical FULL FAT CAN product line
# ------------------------------------------------------------

for expected in expected_low_value_corrections:

    matching_mask = (
        df["TransactionID"]
        .astype("string")
        .str.strip()
        .eq(expected["TransactionID"])

        & pd.to_numeric(
            df["PLUCode"],
            errors="coerce"
        ).eq(expected["PLUCode"])

        & pd.to_numeric(
            df["TransValueCents"],
            errors="coerce"
        ).eq(expected["TransValueCents"])
    )


    matching_row = df.loc[
        matching_mask
    ]


    assert len(matching_row) == 1, (
        "ERROR: Expected exactly one matching product line:\n"
        f"TransactionID: {expected['TransactionID']}\n"
        f"PLUCode: {expected['PLUCode']}\n"
        f"TransValue: €{expected['TransValueCents'] / 100:.2f}\n"
        f"Rows found: {len(matching_row)}"
    )


    actual_product_name = (
        str(
            matching_row[
                "PLUName"
            ].iloc[0]
        )
        .strip()
    )


    assert actual_product_name == "FULL FAT CAN", (
        "ERROR: The matched product line is not "
        "FULL FAT CAN."
    )


    actual_units = int(
        matching_row[
            "UnitSold"
        ].iloc[0]
    )


    assert actual_units == expected["UnitSold"], (
        f"ERROR: Expected UnitSold = "
        f"{expected['UnitSold']} for "
        f"{expected['TransactionID']} at "
        f"€{expected['TransValueCents'] / 100:.2f}, "
        f"but found UnitSold = {actual_units}."
    )


    actual_original_units = int(
        matching_row[
            "UnitSold_Original"
        ].iloc[0]
    )


    assert actual_original_units == 1, (
        "ERROR: The original UnitSold for the historical "
        "FULL FAT CAN line should be 1."
    )


    actual_unit_price_cents = int(
        matching_row[
            "UnitPriceCents"
        ].iloc[0]
    )


    assert actual_unit_price_cents == 180, (
        "ERROR: The matched FULL FAT CAN row does not "
        "have the confirmed €1.80 unit price."
    )


    actual_unit_price = float(
        matching_row[
            "UnitPrice"
        ].iloc[0]
    )


    assert np.isclose(
        actual_unit_price,
        1.80
    ), (
        "ERROR: The matched FULL FAT CAN row does not "
        "have UnitPrice = €1.80."
    )


    inferred_quantity = int(
        matching_row[
            "InferredQuantityFromPrice"
        ].iloc[0]
    )


    assert inferred_quantity == expected["UnitSold"], (
        "ERROR: InferredQuantityFromPrice does not match "
        "the corrected UnitSold."
    )


    assert bool(
        matching_row[
            "IsExactPriceMultiple"
        ].iloc[0]
    ), (
        "ERROR: The matched row is not marked as an exact "
        "price multiple."
    )


    assert bool(
        matching_row[
            "MultiUnitAdjustedFlag"
        ].iloc[0]
    ), (
        "ERROR: The matched row is not marked as adjusted."
    )


print(
    "Both historical FULL FAT CAN product lines "
    "were verified successfully."
)


# ------------------------------------------------------------
# 7. Every adjusted transaction must have valid price data
# ------------------------------------------------------------

assert df.loc[
    adjustment_mask,
    "UnitPriceCents"
].notna().all(), (
    "ERROR: An adjusted row has no UnitPriceCents value."
)


assert pd.to_numeric(
    df.loc[
        adjustment_mask,
        "UnitPriceCents"
    ],
    errors="raise"
).gt(0).all(), (
    "ERROR: An adjusted row has an invalid unit price."
)


assert pd.to_numeric(
    df.loc[
        adjustment_mask,
        "UnitSold"
    ],
    errors="raise"
).gt(1).all(), (
    "ERROR: An adjusted row does not have UnitSold above 1."
)


assert (
    df.loc[
        adjustment_mask,
        "IsExactPriceMultiple"
    ]
    .fillna(False)
    .astype(bool)
    .all()
), (
    "ERROR: An adjusted transaction is not marked as an "
    "exact price multiple."
)


# ------------------------------------------------------------
# 8. Reconstruct every adjusted transaction value
# ------------------------------------------------------------

reconstructed_transaction_cents = (
    pd.to_numeric(
        df.loc[
            adjustment_mask,
            "UnitSold"
        ],
        errors="raise"
    ).to_numpy(dtype="int64")

    *

    pd.to_numeric(
        df.loc[
            adjustment_mask,
            "UnitPriceCents"
        ],
        errors="raise"
    ).to_numpy(dtype="int64")
)


actual_adjusted_transaction_cents = (
    pd.to_numeric(
        df.loc[
            adjustment_mask,
            "TransValueCents"
        ],
        errors="raise"
    )
    .to_numpy(dtype="int64")
)


assert np.array_equal(
    reconstructed_transaction_cents,
    actual_adjusted_transaction_cents
), (
    "ERROR: At least one corrected UnitSold multiplied by "
    "UnitPrice does not equal its TransValue."
)


# ------------------------------------------------------------
# 9. Every non-adjusted row must retain original UnitSold
# ------------------------------------------------------------

current_non_adjusted_units = (
    pd.to_numeric(
        df.loc[
            ~adjustment_mask,
            "UnitSold"
        ],
        errors="raise"
    )
    .astype("Int64")
)


original_non_adjusted_units = (
    pd.to_numeric(
        df.loc[
            ~adjustment_mask,
            "UnitSold_Original"
        ],
        errors="raise"
    )
    .astype("Int64")
)


assert current_non_adjusted_units.equals(
    original_non_adjusted_units
), (
    "ERROR: A transaction not marked for adjustment has "
    "a changed UnitSold value."
)


# ------------------------------------------------------------
# 10. UnitSold must contain positive whole numbers only
# ------------------------------------------------------------

unit_sold_numeric = pd.to_numeric(
    df["UnitSold"],
    errors="raise"
)


assert unit_sold_numeric.notna().all(), (
    "ERROR: UnitSold contains missing values."
)


assert unit_sold_numeric.ge(1).all(), (
    "ERROR: UnitSold contains a value below 1."
)


assert np.allclose(
    unit_sold_numeric.to_numpy(dtype=float),
    np.round(
        unit_sold_numeric.to_numpy(dtype=float)
    )
), (
    "ERROR: UnitSold contains a non-whole-number value."
)


# ------------------------------------------------------------
# 11. Verify that transaction values were not changed
# ------------------------------------------------------------

current_transaction_value_cents = int(
    pd.to_numeric(
        df["TransValueCents"],
        errors="raise"
    ).sum()
)


assert (
    current_transaction_value_cents
    == original_transaction_value_cents
), (
    "ERROR: The total transaction value changed."
)


# ------------------------------------------------------------
# 12. Verify the final expected totals
# ------------------------------------------------------------

original_unit_total = int(
    pd.to_numeric(
        df["UnitSold_Original"],
        errors="raise"
    ).sum()
)


corrected_unit_total = int(
    pd.to_numeric(
        df["UnitSold"],
        errors="raise"
    ).sum()
)


additional_units = (
    corrected_unit_total
    - original_unit_total
)


assert original_unit_total == 138_983, (
    "ERROR: Original UnitSold total should be 138,983."
)


assert corrected_unit_total == 141_480, (
    "ERROR: Corrected UnitSold total should be 141,480."
)


assert additional_units == 2_497, (
    "ERROR: Additional units should total 2,497."
)


# ------------------------------------------------------------
# Final results
# ------------------------------------------------------------

print()
print("All final validation checks passed.")
print()
print("Rows:", f"{len(df):,}")
print(
    "Adjusted transaction rows:",
    f"{adjusted_row_count:,}"
)
print(
    "Original UnitSold total:",
    f"{original_unit_total:,}"
)
print(
    "Corrected UnitSold total:",
    f"{corrected_unit_total:,}"
)
print(
    "Additional units identified:",
    f"{additional_units:,}"
)
print(
    "OPEN UL adjusted:",
    f"{open_ul_adjusted_count:,}"
)
print(
    "Confirmed historical FULL FAT CAN rows "
    "at or below €9.30 adjusted:",
    len(low_value_adjusted_rows)
)

display(low_value_adjusted_rows)

Both historical FULL FAT CAN product lines were verified successfully.

All final validation checks passed.

Rows: 138,983
Adjusted transaction rows: 61
Original UnitSold total: 138,983
Corrected UnitSold total: 141,480
Additional units identified: 2,497
OPEN UL adjusted: 0
Confirmed historical FULL FAT CAN rows at or below €9.30 adjusted: 2


,TransDate,TransactionID,PLUCode,PLUName,GroupName,TransValue,TransValueCents,UnitPrice,UnitPriceCents,UnitSold_Original,UnitSold,InferredQuantityFromPrice,IsExactPriceMultiple,MultiUnitAdjustedFlag,UnitSoldAdjustmentMethod
0,2025-05-29 18:48:00,184434_2025-05-29_18-48-00,2000000019,FULL FAT CAN,COLD BEVS,9.0,900,1.8,180,1,5,5,True,True,Adjusted: historical FULL FAT CAN transaction ...
1,2025-07-24 12:53:00,194858_2025-07-24_12-53-00,2000000019,FULL FAT CAN,COLD BEVS,7.2,720,1.8,180,1,4,4,True,True,Adjusted: historical FULL FAT CAN transaction ...


In [64]:
# ============================================================
# New Cell 11
# Prepare the final corrected datasets
# ============================================================

# Rebuild the final adjustment mask for safety
adjustment_mask = (
    df["MultiUnitAdjustedFlag"]
    .fillna(False)
    .astype(bool)
)


# Confirm final state before preparing files
assert len(df) == 138_983
assert int(adjustment_mask.sum()) == 61
assert int(df["UnitSold"].sum()) == 141_480


# Ensure output folder exists
DATA_FOLDER.mkdir(
    parents=True,
    exist_ok=True
)


# ------------------------------------------------------------
# Main model-ready transaction dataset
# ------------------------------------------------------------
# Keep exactly the original transaction columns.
# The only modified original column is UnitSold.

final_model_ready_df = df[
    original_columns
].copy()


# ------------------------------------------------------------
# Full transaction-level audit dataset
# ------------------------------------------------------------

full_audit_columns = (
    original_columns
    + [
        "UnitSold_Original",
        "UnitPrice",
        "UnitPriceFrequency",
        "InferredQuantityFromPrice",
        "IsExactPriceMultiple",
        "MultiUnitAdjustedFlag",
        "UnitSoldAdjustmentMethod",
    ]
)


missing_full_audit_columns = [
    column
    for column in full_audit_columns
    if column not in df.columns
]


if missing_full_audit_columns:
    raise ValueError(
        "The following required audit columns are missing: "
        f"{missing_full_audit_columns}"
    )


full_audit_df = df[
    full_audit_columns
].copy()


# ------------------------------------------------------------
# Rebuild audit containing only adjusted rows
# ------------------------------------------------------------

adjustment_audit_columns = [
    "TransDate",
    "TransactionID",
    "PLUCode",
    "PLUName",
    "GroupCode",
    "GroupName",
    "TransValue",
    "UnitPrice",
    "UnitPriceFrequency",
    "UnitSold_Original",
    "UnitSold",
    "InferredQuantityFromPrice",
    "IsExactPriceMultiple",
    "MultiUnitAdjustedFlag",
    "UnitSoldAdjustmentMethod",
]


adjustment_audit = (
    df.loc[
        adjustment_mask,
        adjustment_audit_columns,
    ]
    .sort_values(
        [
            "TransValue",
            "PLUName",
            "TransactionID",
        ],
        ascending=[
            False,
            True,
            True,
        ]
    )
    .reset_index(drop=True)
)


# ------------------------------------------------------------
# Final preparation checks
# ------------------------------------------------------------

assert len(final_model_ready_df) == 138_983, (
    "Incorrect model-ready row count."
)

assert (
    final_model_ready_df.columns.tolist()
    == original_columns
), (
    "The model-ready dataset does not contain exactly "
    "the original columns."
)

assert len(full_audit_df) == 138_983, (
    "Incorrect full-audit row count."
)

assert len(adjustment_audit) == 61, (
    "The adjusted-row audit should contain 61 rows."
)

assert int(
    final_model_ready_df["UnitSold"].sum()
) == 141_480, (
    "The prepared model-ready UnitSold total is incorrect."
)


print("Final datasets prepared successfully.")
print()
print(
    "Model-ready rows:",
    f"{len(final_model_ready_df):,}"
)
print(
    "Model-ready columns:",
    len(final_model_ready_df.columns)
)
print(
    "Full-audit rows:",
    f"{len(full_audit_df):,}"
)
print(
    "Adjusted-row audit entries:",
    f"{len(adjustment_audit):,}"
)
print(
    "Corrected UnitSold total:",
    f"{int(final_model_ready_df['UnitSold'].sum()):,}"
)

Final datasets prepared successfully.

Model-ready rows: 138,983
Model-ready columns: 13
Full-audit rows: 138,983
Adjusted-row audit entries: 61
Corrected UnitSold total: 141,480


In [65]:
# ============================================================
# New Cell 12
# Save all corrected files
# ============================================================

# Main corrected transaction dataset for demand aggregation
final_model_ready_df.to_csv(
    FINAL_OUTPUT_FILE,
    index=False
)


# Full transaction dataset with audit information
full_audit_df.to_csv(
    FULL_AUDIT_OUTPUT_FILE,
    index=False
)


# Only the 61 adjusted transaction rows
adjustment_audit.to_csv(
    ADJUSTED_ROWS_AUDIT_FILE,
    index=False
)


# Product-level adjustment summary
adjustment_summary.to_csv(
    ADJUSTMENT_SUMMARY_FILE,
    index=False
)


# Selected unit-price reference
unit_price_reference.to_csv(
    UNIT_PRICE_REFERENCE_FILE,
    index=False
)


# All unit-price candidates used as evidence
unit_price_candidate_evidence.to_csv(
    UNIT_PRICE_EVIDENCE_FILE,
    index=False
)


print("All corrected files were saved successfully.")
print()

print("1. Main corrected model-ready dataset:")
print(FINAL_OUTPUT_FILE)
print()

print("2. Full transaction audit dataset:")
print(FULL_AUDIT_OUTPUT_FILE)
print()

print("3. Adjusted transaction-row audit:")
print(ADJUSTED_ROWS_AUDIT_FILE)
print()

print("4. Product-level adjustment summary:")
print(ADJUSTMENT_SUMMARY_FILE)
print()

print("5. Unit-price reference:")
print(UNIT_PRICE_REFERENCE_FILE)
print()

print("6. Unit-price evidence:")
print(UNIT_PRICE_EVIDENCE_FILE)

All corrected files were saved successfully.

1. Main corrected model-ready dataset:
eden_datasets/UL_EDEN_clean_final_model_ready_transactions_unitsold_corrected.csv

2. Full transaction audit dataset:
eden_datasets/UL_EDEN_transactions_unitsold_corrected_with_audit_columns.csv

3. Adjusted transaction-row audit:
eden_datasets/unit_sold_adjustment_audit_above_9_30_excluding_open_ul.csv

4. Product-level adjustment summary:
eden_datasets/unit_sold_adjustment_summary_above_9_30.csv

5. Unit-price reference:
eden_datasets/unit_price_reference_for_bulk_transactions.csv

6. Unit-price evidence:
eden_datasets/unit_price_candidate_evidence_for_bulk_transactions.csv


In [67]:
# ============================================================
# New Cell 13
# Confirm that all output files were created
# ============================================================

output_files = {
    "Main model-ready dataset":
        FINAL_OUTPUT_FILE,

    "Full transaction audit dataset":
        FULL_AUDIT_OUTPUT_FILE,

    "Adjusted-row audit":
        ADJUSTED_ROWS_AUDIT_FILE,

    "Adjustment summary":
        ADJUSTMENT_SUMMARY_FILE,

    "Unit-price reference":
        UNIT_PRICE_REFERENCE_FILE,

    "Unit-price evidence":
        UNIT_PRICE_EVIDENCE_FILE,
}


missing_output_files = [
    file_path
    for file_path in output_files.values()
    if not file_path.exists()
]


if missing_output_files:
    missing_text = "\n".join(
        str(file_path)
        for file_path in missing_output_files
    )

    raise FileNotFoundError(
        "The following expected output files were not created:\n"
        f"{missing_text}"
    )


print("All expected output files exist.")
print()


for description, file_path in output_files.items():

    file_size_mb = (
        file_path.stat().st_size
        / (1024 ** 2)
    )

    print(
        f"{description}: "
        f"{file_path} "
        f"({file_size_mb:.2f} MB)"
    )

All expected output files exist.

Main model-ready dataset: eden_datasets/UL_EDEN_clean_final_model_ready_transactions_unitsold_corrected.csv (14.79 MB)
Full transaction audit dataset: eden_datasets/UL_EDEN_transactions_unitsold_corrected_with_audit_columns.csv (23.65 MB)
Adjusted-row audit: eden_datasets/unit_sold_adjustment_audit_above_9_30_excluding_open_ul.csv (0.01 MB)
Adjustment summary: eden_datasets/unit_sold_adjustment_summary_above_9_30.csv (0.00 MB)
Unit-price reference: eden_datasets/unit_price_reference_for_bulk_transactions.csv (0.00 MB)
Unit-price evidence: eden_datasets/unit_price_candidate_evidence_for_bulk_transactions.csv (0.00 MB)


In [68]:
# ============================================================
# New Cell 14
# Reload the saved main model-ready dataset
# ============================================================

saved_model_df = pd.read_csv(
    FINAL_OUTPUT_FILE
)


print("Saved model-ready dataset reloaded.")
print()
print("File:", FINAL_OUTPUT_FILE)
print("Rows:", f"{len(saved_model_df):,}")
print("Columns:", len(saved_model_df.columns))

display(saved_model_df.head())

Saved model-ready dataset reloaded.

File: eden_datasets/UL_EDEN_clean_final_model_ready_transactions_unitsold_corrected.csv
Rows: 138,983
Columns: 13


,TransDate,TransValue,PLUName,GroupCode,GroupName,PLUCode,Date,Hour,DayOfWeek,Month,WeekOfYear,TransactionID,UnitSold
0,2025-07-16 14:46:00,1026.0,VEGT MAINS 3,7,DINNER,4241483,2025-07-16,14,Wednesday,7,29,194814_2025-07-16_14-46-00,114
1,2025-07-17 14:14:00,549.0,MAINS 3,7,DINNER,4241480,2025-07-17,14,Thursday,7,29,194832_2025-07-17_14-14-00,61
2,2025-07-09 14:27:00,513.0,MAINS 3,7,DINNER,4241480,2025-07-09,14,Wednesday,7,28,194782_2025-07-09_14-27-00,57
3,2025-07-18 14:35:00,513.0,MAINS 3,7,DINNER,4241480,2025-07-18,14,Friday,7,29,194837_2025-07-18_14-35-00,57
4,2025-07-10 14:51:00,468.0,MAINS 3,7,DINNER,4241480,2025-07-10,14,Thursday,7,28,194801_2025-07-10_14-51-00,52


In [69]:
# ============================================================
# New Cell 15
# Validate the saved main model-ready dataset
# ============================================================

# Convert important columns safely
saved_model_df["TransValue"] = pd.to_numeric(
    saved_model_df["TransValue"],
    errors="raise"
)


saved_model_df["PLUCode"] = pd.to_numeric(
    saved_model_df["PLUCode"],
    errors="raise"
)


saved_model_df["UnitSold"] = (
    pd.to_numeric(
        saved_model_df["UnitSold"],
        errors="raise"
    )
    .round()
    .astype("Int64")
)


# ------------------------------------------------------------
# 1. Structure checks
# ------------------------------------------------------------

assert len(saved_model_df) == 138_983, (
    "ERROR: Saved row count is incorrect."
)


assert (
    saved_model_df.columns.tolist()
    == original_columns
), (
    "ERROR: Saved model-ready columns do not match "
    "the original transaction columns."
)


# ------------------------------------------------------------
# 2. UnitSold checks
# ------------------------------------------------------------

assert saved_model_df["UnitSold"].notna().all(), (
    "ERROR: Saved UnitSold contains missing values."
)


assert saved_model_df["UnitSold"].ge(1).all(), (
    "ERROR: Saved UnitSold contains a value below 1."
)


assert int(
    saved_model_df["UnitSold"].sum()
) == 141_480, (
    "ERROR: Saved UnitSold total should be 141,480."
)


saved_adjusted_row_count = int(
    saved_model_df["UnitSold"].gt(1).sum()
)


assert saved_adjusted_row_count == 61, (
    "ERROR: Exactly 61 saved transaction rows should "
    "have UnitSold greater than 1."
)


# ------------------------------------------------------------
# 3. OPEN UL must remain UnitSold = 1
# ------------------------------------------------------------

saved_open_ul_mask = (
    saved_model_df["PLUName"]
    .astype("string")
    .str.strip()
    .str.fullmatch(
        r"OPEN\s+UL",
        case=False,
        na=False
    )
)


assert saved_model_df.loc[
    saved_open_ul_mask,
    "UnitSold"
].eq(1).all(), (
    "ERROR: At least one OPEN UL row changed."
)


# ------------------------------------------------------------
# 4. Verify the two historical FULL FAT CAN corrections
# ------------------------------------------------------------

saved_expected_full_fat_can_corrections = [
    {
        "TransactionID": "184434_2025-05-29_18-48-00",
        "PLUCode": 2000000019,
        "TransValue": 9.00,
        "UnitSold": 5,
    },
    {
        "TransactionID": "194858_2025-07-24_12-53-00",
        "PLUCode": 2000000019,
        "TransValue": 7.20,
        "UnitSold": 4,
    },
]


for expected in saved_expected_full_fat_can_corrections:

    matching_mask = (
        saved_model_df["TransactionID"]
        .astype("string")
        .str.strip()
        .eq(expected["TransactionID"])

        & saved_model_df["PLUCode"]
        .eq(expected["PLUCode"])

        & np.isclose(
            saved_model_df["TransValue"],
            expected["TransValue"],
            atol=0.001
        )
    )


    matching_row = saved_model_df.loc[
        matching_mask
    ]


    assert len(matching_row) == 1, (
        "ERROR: Expected exactly one saved product line:\n"
        f"TransactionID: {expected['TransactionID']}\n"
        f"PLUCode: {expected['PLUCode']}\n"
        f"TransValue: €{expected['TransValue']:.2f}\n"
        f"Rows found: {len(matching_row)}"
    )


    assert (
        matching_row["PLUName"]
        .astype("string")
        .str.strip()
        .eq("FULL FAT CAN")
        .all()
    ), (
        "ERROR: The matching saved row is not "
        "FULL FAT CAN."
    )


    actual_units = int(
        matching_row["UnitSold"].iloc[0]
    )


    assert actual_units == expected["UnitSold"], (
        f"ERROR: Expected UnitSold = "
        f"{expected['UnitSold']} for "
        f"{expected['TransactionID']} at "
        f"€{expected['TransValue']:.2f}, "
        f"but found {actual_units}."
    )


# ------------------------------------------------------------
# 5. Verify all seven corrected FULL FAT CAN rows
# ------------------------------------------------------------

saved_full_fat_can_corrected_mask = (
    saved_model_df["PLUCode"].eq(2000000019)
    & saved_model_df["PLUName"]
        .astype("string")
        .str.strip()
        .eq("FULL FAT CAN")
    & saved_model_df["UnitSold"].gt(1)
)


saved_full_fat_can_corrected = (
    saved_model_df.loc[
        saved_full_fat_can_corrected_mask,
        [
            "TransDate",
            "TransactionID",
            "PLUCode",
            "PLUName",
            "TransValue",
            "UnitSold",
        ],
    ]
    .sort_values(
        "TransValue",
        ascending=False
    )
    .reset_index(drop=True)
)


assert len(saved_full_fat_can_corrected) == 7, (
    "ERROR: Expected seven corrected FULL FAT CAN rows."
)


assert int(
    saved_full_fat_can_corrected["UnitSold"].sum()
) == 93, (
    "ERROR: Corrected FULL FAT CAN units should total 93."
)


# ------------------------------------------------------------
# 6. Verify total transaction value remained unchanged
# ------------------------------------------------------------

saved_transaction_value_cents = int(
    (
        saved_model_df["TransValue"]
        * 100
    )
    .round()
    .sum()
)


assert (
    saved_transaction_value_cents
    == original_transaction_value_cents
), (
    "ERROR: The saved total transaction value changed."
)


print("Saved main model-ready dataset validation passed.")
print()
print("Rows:", f"{len(saved_model_df):,}")
print(
    "Rows with UnitSold above 1:",
    f"{saved_adjusted_row_count:,}"
)
print(
    "Corrected UnitSold total:",
    f"{int(saved_model_df['UnitSold'].sum()):,}"
)
print(
    "OPEN UL rows unchanged:",
    f"{int(saved_open_ul_mask.sum()):,}"
)
print(
    "Corrected FULL FAT CAN rows:",
    len(saved_full_fat_can_corrected)
)
print(
    "Corrected FULL FAT CAN units:",
    int(saved_full_fat_can_corrected["UnitSold"].sum())
)

display(saved_full_fat_can_corrected)

Saved main model-ready dataset validation passed.

Rows: 138,983
Rows with UnitSold above 1: 61
Corrected UnitSold total: 141,480
OPEN UL rows unchanged: 24,736
Corrected FULL FAT CAN rows: 7
Corrected FULL FAT CAN units: 93


,TransDate,TransactionID,PLUCode,PLUName,TransValue,UnitSold
0,2025-07-09 14:27:00,194782_2025-07-09_14-27-00,2000000019,FULL FAT CAN,48.6,27
1,2025-07-08 15:13:00,194769_2025-07-08_15-13-00,2000000019,FULL FAT CAN,46.8,26
2,2025-07-25 12:51:00,194864_2025-07-25_12-51-00,2000000019,FULL FAT CAN,19.8,11
3,2025-07-24 12:53:00,194858_2025-07-24_12-53-00,2000000019,FULL FAT CAN,18.0,10
4,2025-07-25 12:51:00,194864_2025-07-25_12-51-00,2000000019,FULL FAT CAN,18.0,10
5,2025-05-29 18:48:00,184434_2025-05-29_18-48-00,2000000019,FULL FAT CAN,9.0,5
6,2025-07-24 12:53:00,194858_2025-07-24_12-53-00,2000000019,FULL FAT CAN,7.2,4


In [70]:
# ============================================================
# New Cell 16
# Reload and validate the full transaction audit dataset
# ============================================================

saved_full_audit_df = pd.read_csv(
    FULL_AUDIT_OUTPUT_FILE
)


# ------------------------------------------------------------
# Helper for safely converting CSV Boolean columns
# ------------------------------------------------------------

def convert_csv_boolean(series):
    """
    Convert CSV text values such as 'True' and 'False'
    into proper Boolean values.
    """

    normalized = (
        series
        .astype("string")
        .str.strip()
        .str.lower()
    )


    converted = normalized.map({
        "true": True,
        "false": False,
    })


    if converted.isna().any():

        invalid_values = (
            series.loc[
                converted.isna()
            ]
            .drop_duplicates()
            .tolist()
        )

        raise ValueError(
            "Invalid Boolean values found in saved file: "
            f"{invalid_values}"
        )


    return converted.astype(bool)


# Convert saved columns
saved_full_audit_df[
    "MultiUnitAdjustedFlag"
] = convert_csv_boolean(
    saved_full_audit_df[
        "MultiUnitAdjustedFlag"
    ]
)


saved_full_audit_df[
    "IsExactPriceMultiple"
] = convert_csv_boolean(
    saved_full_audit_df[
        "IsExactPriceMultiple"
    ]
)


saved_full_audit_df["UnitSold"] = (
    pd.to_numeric(
        saved_full_audit_df["UnitSold"],
        errors="raise"
    )
    .round()
    .astype("Int64")
)


saved_full_audit_df["UnitSold_Original"] = (
    pd.to_numeric(
        saved_full_audit_df["UnitSold_Original"],
        errors="raise"
    )
    .round()
    .astype("Int64")
)


saved_full_audit_df["UnitPrice"] = pd.to_numeric(
    saved_full_audit_df["UnitPrice"],
    errors="coerce"
)


saved_full_audit_df[
    "InferredQuantityFromPrice"
] = (
    pd.to_numeric(
        saved_full_audit_df[
            "InferredQuantityFromPrice"
        ],
        errors="coerce"
    )
    .round()
    .astype("Int64")
)


saved_audit_adjustment_mask = (
    saved_full_audit_df[
        "MultiUnitAdjustedFlag"
    ]
)


# ------------------------------------------------------------
# Validate full audit dataset
# ------------------------------------------------------------

assert len(saved_full_audit_df) == 138_983, (
    "ERROR: Full-audit row count is incorrect."
)


assert int(
    saved_audit_adjustment_mask.sum()
) == 61, (
    "ERROR: Full-audit file should contain exactly "
    "61 adjusted rows."
)


assert int(
    saved_full_audit_df["UnitSold"].sum()
) == 141_480, (
    "ERROR: Full-audit UnitSold total is incorrect."
)


assert int(
    saved_full_audit_df[
        "UnitSold_Original"
    ].sum()
) == 138_983, (
    "ERROR: Full-audit original UnitSold total is incorrect."
)


assert saved_full_audit_df.loc[
    saved_audit_adjustment_mask,
    "IsExactPriceMultiple"
].all(), (
    "ERROR: An adjusted audit row is not marked as an "
    "exact price multiple."
)


assert saved_full_audit_df.loc[
    saved_audit_adjustment_mask,
    "UnitPrice"
].notna().all(), (
    "ERROR: An adjusted audit row has no unit price."
)


assert saved_full_audit_df.loc[
    saved_audit_adjustment_mask,
    "UnitSold"
].gt(1).all(), (
    "ERROR: An adjusted audit row has UnitSold <= 1."
)


# Non-adjusted rows must remain unchanged
assert (
    saved_full_audit_df.loc[
        ~saved_audit_adjustment_mask,
        "UnitSold"
    ]
    .equals(
        saved_full_audit_df.loc[
            ~saved_audit_adjustment_mask,
            "UnitSold_Original"
        ]
    )
), (
    "ERROR: A non-adjusted audit row has a changed UnitSold."
)


# OPEN UL audit check
saved_audit_open_ul_mask = (
    saved_full_audit_df["PLUName"]
    .astype("string")
    .str.strip()
    .str.fullmatch(
        r"OPEN\s+UL",
        case=False,
        na=False
    )
)


assert not saved_full_audit_df.loc[
    saved_audit_open_ul_mask,
    "MultiUnitAdjustedFlag"
].any(), (
    "ERROR: An OPEN UL row was adjusted in the saved "
    "audit dataset."
)


# Verify the two low-value adjusted lines
saved_low_value_adjusted_mask = (
    saved_audit_adjustment_mask
    & pd.to_numeric(
        saved_full_audit_df["TransValue"],
        errors="raise"
    ).le(9.30)
)


saved_low_value_adjusted_rows = (
    saved_full_audit_df.loc[
        saved_low_value_adjusted_mask,
        [
            "TransactionID",
            "PLUCode",
            "PLUName",
            "TransValue",
            "UnitPrice",
            "UnitSold_Original",
            "UnitSold",
            "MultiUnitAdjustedFlag",
        ],
    ]
    .sort_values(
        "TransValue",
        ascending=False
    )
    .reset_index(drop=True)
)


assert len(saved_low_value_adjusted_rows) == 2, (
    "ERROR: Full-audit file should contain exactly two "
    "adjusted rows at or below €9.30."
)


assert saved_low_value_adjusted_rows[
    "PLUName"
].astype("string").str.strip().eq(
    "FULL FAT CAN"
).all(), (
    "ERROR: A non-FULL FAT CAN row was adjusted at or "
    "below €9.30."
)


print("Saved full-audit dataset validation passed.")
print()
print(
    "Full-audit rows:",
    f"{len(saved_full_audit_df):,}"
)
print(
    "Adjusted rows:",
    f"{int(saved_audit_adjustment_mask.sum()):,}"
)
print(
    "Original UnitSold total:",
    f"{int(saved_full_audit_df['UnitSold_Original'].sum()):,}"
)
print(
    "Corrected UnitSold total:",
    f"{int(saved_full_audit_df['UnitSold'].sum()):,}"
)
print(
    "Adjusted rows at or below €9.30:",
    len(saved_low_value_adjusted_rows)
)

display(saved_low_value_adjusted_rows)

Saved full-audit dataset validation passed.

Full-audit rows: 138,983
Adjusted rows: 61
Original UnitSold total: 138,983
Corrected UnitSold total: 141,480
Adjusted rows at or below €9.30: 2


,TransactionID,PLUCode,PLUName,TransValue,UnitPrice,UnitSold_Original,UnitSold,MultiUnitAdjustedFlag
0,184434_2025-05-29_18-48-00,2000000019,FULL FAT CAN,9.0,1.8,1,5,True
1,194858_2025-07-24_12-53-00,2000000019,FULL FAT CAN,7.2,1.8,1,4,True


In [71]:
# ============================================================
# New Cell 17
# Reload and validate the adjusted-row audit
# ============================================================

saved_adjusted_rows_audit = pd.read_csv(
    ADJUSTED_ROWS_AUDIT_FILE
)


saved_adjusted_rows_audit["UnitSold"] = (
    pd.to_numeric(
        saved_adjusted_rows_audit["UnitSold"],
        errors="raise"
    )
    .round()
    .astype("Int64")
)


saved_adjusted_rows_audit[
    "UnitSold_Original"
] = (
    pd.to_numeric(
        saved_adjusted_rows_audit[
            "UnitSold_Original"
        ],
        errors="raise"
    )
    .round()
    .astype("Int64")
)


saved_adjusted_rows_audit["UnitPrice"] = pd.to_numeric(
    saved_adjusted_rows_audit["UnitPrice"],
    errors="raise"
)


saved_adjusted_rows_audit["TransValue"] = pd.to_numeric(
    saved_adjusted_rows_audit["TransValue"],
    errors="raise"
)


assert len(saved_adjusted_rows_audit) == 61, (
    "ERROR: Adjusted-row audit should contain 61 rows."
)


assert saved_adjusted_rows_audit[
    "UnitSold"
].gt(1).all(), (
    "ERROR: Adjusted-row audit contains UnitSold <= 1."
)


assert saved_adjusted_rows_audit[
    "UnitSold_Original"
].eq(1).all(), (
    "ERROR: Adjusted-row audit contains an unexpected "
    "original UnitSold value."
)


audit_reconstructed_values = (
    saved_adjusted_rows_audit["UnitSold"].astype(float)
    * saved_adjusted_rows_audit["UnitPrice"]
).round(2)


assert np.allclose(
    audit_reconstructed_values,
    saved_adjusted_rows_audit["TransValue"],
    atol=0.001
), (
    "ERROR: An adjusted-row audit value cannot be "
    "reconstructed from UnitSold × UnitPrice."
)


saved_audit_full_fat_can = (
    saved_adjusted_rows_audit.loc[
        saved_adjusted_rows_audit["PLUName"]
        .astype("string")
        .str.strip()
        .eq("FULL FAT CAN")
    ]
    .sort_values(
        "TransValue",
        ascending=False
    )
    .reset_index(drop=True)
)


assert len(saved_audit_full_fat_can) == 7, (
    "ERROR: Adjusted-row audit should contain seven "
    "FULL FAT CAN rows."
)


assert int(
    saved_audit_full_fat_can["UnitSold"].sum()
) == 93, (
    "ERROR: FULL FAT CAN corrected units should total 93."
)


print("Saved adjusted-row audit validation passed.")
print()
print(
    "Audit rows:",
    len(saved_adjusted_rows_audit)
)
print(
    "FULL FAT CAN audit rows:",
    len(saved_audit_full_fat_can)
)
print(
    "FULL FAT CAN corrected units:",
    int(saved_audit_full_fat_can["UnitSold"].sum())
)

display(saved_audit_full_fat_can)

Saved adjusted-row audit validation passed.

Audit rows: 61
FULL FAT CAN audit rows: 7
FULL FAT CAN corrected units: 93


,TransDate,TransactionID,PLUCode,PLUName,GroupCode,GroupName,TransValue,UnitPrice,UnitPriceFrequency,UnitSold_Original,UnitSold,InferredQuantityFromPrice,IsExactPriceMultiple,MultiUnitAdjustedFlag,UnitSoldAdjustmentMethod
0,2025-07-09 14:27:00,194782_2025-07-09_14-27-00,2000000019,FULL FAT CAN,2,COLD BEVS,48.6,1.8,789,1,27,27,True,True,Adjusted: TransValue > €9.30 and exact multipl...
1,2025-07-08 15:13:00,194769_2025-07-08_15-13-00,2000000019,FULL FAT CAN,2,COLD BEVS,46.8,1.8,789,1,26,26,True,True,Adjusted: TransValue > €9.30 and exact multipl...
2,2025-07-25 12:51:00,194864_2025-07-25_12-51-00,2000000019,FULL FAT CAN,2,COLD BEVS,19.8,1.8,789,1,11,11,True,True,Adjusted: TransValue > €9.30 and exact multipl...
3,2025-07-24 12:53:00,194858_2025-07-24_12-53-00,2000000019,FULL FAT CAN,2,COLD BEVS,18.0,1.8,789,1,10,10,True,True,Adjusted: TransValue > €9.30 and exact multipl...
4,2025-07-25 12:51:00,194864_2025-07-25_12-51-00,2000000019,FULL FAT CAN,2,COLD BEVS,18.0,1.8,789,1,10,10,True,True,Adjusted: TransValue > €9.30 and exact multipl...
5,2025-05-29 18:48:00,184434_2025-05-29_18-48-00,2000000019,FULL FAT CAN,2,COLD BEVS,9.0,1.8,789,1,5,5,True,True,Adjusted: historical FULL FAT CAN transaction ...
6,2025-07-24 12:53:00,194858_2025-07-24_12-53-00,2000000019,FULL FAT CAN,2,COLD BEVS,7.2,1.8,789,1,4,4,True,True,Adjusted: historical FULL FAT CAN transaction ...


In [72]:
# ============================================================
# New Cell 18
# Final completion summary
# ============================================================

final_original_units = int(
    df["UnitSold_Original"].sum()
)

final_corrected_units = int(
    df["UnitSold"].sum()
)

final_additional_units = (
    final_corrected_units
    - final_original_units
)


print("=" * 72)
print("EDEN TRANSACTION UNIT-SOLD CORRECTION COMPLETED")
print("=" * 72)
print()

print(
    "Transaction rows:",
    f"{len(df):,}"
)

print(
    "Adjusted transaction rows:",
    f"{int(adjustment_mask.sum()):,}"
)

print(
    "Original UnitSold total:",
    f"{final_original_units:,}"
)

print(
    "Corrected UnitSold total:",
    f"{final_corrected_units:,}"
)

print(
    "Additional units identified:",
    f"{final_additional_units:,}"
)

print(
    "OPEN UL rows adjusted:",
    f"{int(adjustment_mask.loc[is_open_ul].sum()):,}"
)

print(
    "FULL FAT CAN adjusted rows:",
    f"{len(saved_audit_full_fat_can):,}"
)

print(
    "FULL FAT CAN corrected units:",
    f"{int(saved_audit_full_fat_can['UnitSold'].sum()):,}"
)

print()
print("Final model-ready transaction file:")
print(FINAL_OUTPUT_FILE)

print()
print(
    "Use this file as the source for creating the "
    "daily product-demand dataset."
)

EDEN TRANSACTION UNIT-SOLD CORRECTION COMPLETED

Transaction rows: 138,983
Adjusted transaction rows: 61
Original UnitSold total: 138,983
Corrected UnitSold total: 141,480
Additional units identified: 2,497
OPEN UL rows adjusted: 0
FULL FAT CAN adjusted rows: 7
FULL FAT CAN corrected units: 93

Final model-ready transaction file:
eden_datasets/UL_EDEN_clean_final_model_ready_transactions_unitsold_corrected.csv

Use this file as the source for creating the daily product-demand dataset.
